# Paper Financial Comparison Plots

This notebook creates separate publication-quality financial figures from saved `daily_financial_detail.csv` files. Optimization notebooks only save CSV outputs; all paper figures are centralized here.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib import ticker
from matplotlib import dates as mdates

PROJECT = Path.cwd()
OUT_DIR = PROJECT / "Paper_Figures" / "financial_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 9.5,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SCENARIOS = [
    ("Shrinking", "Perfect", "Shrinking-Perfect", PROJECT / "Results_Shrinking/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv"),
    ("Shrinking", "Persistence", "Shrinking-Persistence", PROJECT / "Results_Shrinking/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv"),
    ("Rolling", "Perfect", "Rolling-Perfect", PROJECT / "Validation_Results/June_2025/formal_runs/gate_e_20260726_c8250f11/rolling/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv"),
    ("Rolling", "Persistence", "Rolling-Persistence", PROJECT / "Validation_Results/June_2025/formal_runs/gate_e_20260726_c8250f11/rolling/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv"),
]
RETAIL_ONLY_SOURCES = {
    "Shrinking-Perfect": PROJECT / "Results_Shrinking/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv",
    "Shrinking-Persistence": PROJECT / "Results_Shrinking/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv",
    "Rolling-Perfect": PROJECT / "Results_Rolling/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv",
    "Rolling-Persistence": PROJECT / "Results_Rolling/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv",
}
SCENARIO_ORDER = [s[2] for s in SCENARIOS]
CASE_ORDER = ["Retail only", "Both"]
CASE_COLORS = {"Retail only": "#D55E00", "Both": "#0072B2"}
COMPONENT_COLORS = {
    "EV Revenue": "#009E73",
    "WM Revenue": "#0072B2",
    "TOU Cost": "#CC79A7",
    "PD Cost": "#E69F00",
    "NCD Cost": "#6A3D9A",
}
METRICS = ["Total Revenue", "WM Revenue", "TOU Cost", "PD Cost", "NCD Cost", "EV Revenue"]
DECOMP_GROUP_COLORS = {
    "WM EV": "#0072B2",
    "WM BESS": "#009E73",
    "NWM EV": "#E69F00",
    "NWM BESS": "#6A3D9A",
}
DECOMP_COMPONENTS = [
    "WM EV Energy", "WM EV Capacity",
    "WM BESS Energy", "WM BESS Capacity",
    "NWM EV Energy", "NWM EV Capacity",
    "NWM BESS Energy", "NWM BESS Capacity",
]

def dollars(x, pos=None):
    sign = "-" if x < 0 else ""
    x = abs(x)
    if x >= 1000:
        return f"{sign}${x/1000:.0f}k"
    return f"{sign}${x:.0f}"

def save_fig(fig, filename):
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(path.relative_to(PROJECT))
    return path


Matplotlib is building the font cache; this may take a moment.


In [2]:
frames = []
missing = []
for mpc, forecast, scenario, path in SCENARIOS:
    if not path.exists():
        missing.append(path.relative_to(PROJECT))
        continue
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df[df["Case"] == "Both"].copy()
    both_dates = set(df["Date"])
    retail_path = RETAIL_ONLY_SOURCES[scenario]
    if retail_path.exists():
        retail_df = pd.read_csv(retail_path)
        retail_df["Date"] = pd.to_datetime(retail_df["Date"])
        retail_df = retail_df[(retail_df["Case"] == "Retail only") & retail_df["Date"].isin(both_dates)].copy()
        retail_counts = retail_df.groupby("Date").size()
        retail_complete = bool(both_dates) and set(retail_counts.index) == both_dates and bool((retail_counts == 1).all())
        retail_fresh = retail_path.stat().st_mtime >= path.stat().st_mtime
        if retail_complete and retail_fresh:
            df = pd.concat([df, retail_df], ignore_index=True, sort=False)
        elif retail_complete and not retail_fresh:
            print(f"Ignoring stale Retail-only source for {scenario}; rerun that Retail-only case before plotting.")
        elif not retail_df.empty:
            print(f"Ignoring incomplete Retail-only rows for {scenario}: {retail_df['Date'].nunique()} of {len(both_dates)} formal dates.")
    df["MPC"] = mpc
    df["Forecast"] = forecast
    df["Scenario"] = scenario
    frames.append(df)

if missing:
    print("Missing input files:")
    for item in missing:
        print(" -", item)
if not frames:
    raise FileNotFoundError("No daily_financial_detail.csv files found.")

all_df = pd.concat(frames, ignore_index=True)
all_df = all_df[all_df["Case"].isin(CASE_ORDER)].copy()
for col in METRICS + ["WM BESS Total", "WM EV Total", "WM Energy Revenue", "WM Capacity Revenue"] + DECOMP_COMPONENTS:
    if col in all_df.columns:
        all_df[col] = pd.to_numeric(all_df[col], errors="coerce")

summary = all_df.groupby(["Scenario", "MPC", "Forecast", "Case"], as_index=False)[METRICS].sum(numeric_only=True)

wide = all_df.pivot_table(index=["Scenario", "MPC", "Forecast", "Date"], columns="Case", values=METRICS, aggfunc="sum")
delta_df = pd.DataFrame(index=wide.index).reset_index()
for metric in METRICS:
    if (metric, "Both") in wide.columns and (metric, "Retail only") in wide.columns:
        delta_df[metric] = (wide[(metric, "Both")] - wide[(metric, "Retail only")]).values

print(f"Loaded {len(all_df)} case-day rows from {len(frames)} scenario files.")
print(f"Date range: {all_df['Date'].min().date()} to {all_df['Date'].max().date()}")
summary

Loaded 240 case-day rows from 4 scenario files.
Date range: 2025-06-01 to 2025-06-30


,Scenario,MPC,Forecast,Case,Total Revenue,WM Revenue,TOU Cost,PD Cost,NCD Cost,EV Revenue
0,Rolling-Perfect,Rolling,Perfect,Both,10733.15,4347.03,-12300.80,-279.25,-4427.75,23393.92
1,Rolling-Perfect,Rolling,Perfect,Retail only,6240.92,0.00,-12414.51,-310.73,-4427.74,23393.92
2,Rolling-Persistence,Rolling,Persistence,Both,10485.28,4421.87,-11092.27,-467.84,-5770.41,23393.92
3,Rolling-Persistence,Rolling,Persistence,Retail only,6549.50,0.00,-11631.63,-275.30,-4937.48,23393.92
4,Shrinking-Perfect,Shrinking,Perfect,Both,10549.33,4309.08,-12415.17,-310.73,-4427.74,23393.92
5,Shrinking-Perfect,Shrinking,Perfect,Retail only,6240.92,0.00,-12414.51,-310.73,-4427.74,23393.92
6,Shrinking-Persistence,Shrinking,Persistence,Both,11131.92,4414.40,-11438.40,-494.02,-4743.99,23393.92
7,Shrinking-Persistence,Shrinking,Persistence,Retail only,5356.72,0.00,-11611.26,-492.94,-5932.99,23393.92


## Period-aware paper figures

Generate combined and per-month financial figures from the currently saved dispatch results.


In [3]:
# Period-aware financial analysis for the current saved run.
# This cell uses the already-loaded all_df and regenerates paper-quality figures for:
#   1) the full available date range, and
#   2) each calendar month present in the result CSVs.

PERIOD_ROOT = OUT_DIR
PERIOD_ROOT.mkdir(parents=True, exist_ok=True)


def _period_label(df):
    start = pd.Timestamp(df["Date"].min()).strftime("%Y-%m-%d")
    end = pd.Timestamp(df["Date"].max()).strftime("%Y-%m-%d")
    return f"{start}_to_{end}"


def _make_period_summary(df, out_dir):
    period_summary = df.groupby(["Scenario", "MPC", "Forecast", "Case"], as_index=False)[METRICS].sum(numeric_only=True)
    period_summary.to_csv(out_dir / "financial_summary_by_scenario_case.csv", index=False)

    wide = df.pivot_table(index=["Scenario", "MPC", "Forecast", "Date"], columns="Case", values=METRICS, aggfunc="sum")
    period_delta = pd.DataFrame(index=wide.index).reset_index()
    for metric in METRICS:
        if (metric, "Both") in wide.columns and (metric, "Retail only") in wide.columns:
            period_delta[metric] = (wide[(metric, "Both")] - wide[(metric, "Retail only")]).values
    period_delta.to_csv(out_dir / "financial_delta_both_minus_retail_daily.csv", index=False)
    return period_summary, period_delta


def _save_period_fig(fig, out_dir, filename):
    path = out_dir / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(path.relative_to(PROJECT))
    return path


def plot_period_financials(period_df, label, title_suffix):
    out_dir = PERIOD_ROOT / label
    out_dir.mkdir(parents=True, exist_ok=True)
    period_df = period_df.copy()
    available_cases = set(period_df['Case'].dropna())
    paired_cases_complete = True
    for scenario in SCENARIO_ORDER:
        scenario_df = period_df[period_df['Scenario'] == scenario]
        per_date_cases = scenario_df.groupby('Date')['Case'].agg(set)
        if scenario_df.empty or per_date_cases.empty or not per_date_cases.map(lambda cases: set(CASE_ORDER).issubset(cases)).all():
            paired_cases_complete = False
            break
    if not paired_cases_complete:
        period_summary = period_df.groupby(['Scenario', 'MPC', 'Forecast', 'Case'], as_index=False)[METRICS].sum(numeric_only=True)
        period_summary.to_csv(out_dir / 'financial_summary_by_scenario_case.csv', index=False)
        stale_delta = out_dir / 'financial_delta_both_minus_retail_daily.csv'
        if stale_delta.exists():
            stale_delta.unlink()
        stale_paired_figures = [
            'fig_financial_violin_daily_total.png',
            'fig_financial_delta_heatmap.png',
            'fig_financial_case_comparison_subplots.png',
            'fig_financial_case_value_stack_subplots.png',
            'fig_profit_decomposition_daily.png',
            'fig_profit_decomposition_period_total.png',
        ]
        for stale_name in stale_paired_figures:
            stale_path = out_dir / stale_name
            if stale_path.exists():
                stale_path.unlink()
        if all(col in period_df.columns for col in DECOMP_COMPONENTS):
            both_decomp = period_df[period_df['Case'] == 'Both'][['Scenario', 'Date', 'Total Revenue'] + DECOMP_COMPONENTS].copy()
            both_decomp = both_decomp.sort_values(['Scenario', 'Date'])
            both_decomp['Decomposition Residual'] = both_decomp['Total Revenue'] - both_decomp[DECOMP_COMPONENTS].sum(axis=1)
            both_decomp.to_csv(out_dir / 'profit_decomposition_daily.csv', index=False)
            decomp_total = both_decomp.groupby('Scenario', as_index=False)[['Total Revenue'] + DECOMP_COMPONENTS + ['Decomposition Residual']].sum(numeric_only=True)
            decomp_total = decomp_total.set_index('Scenario').reindex(SCENARIO_ORDER).dropna(how='all').reset_index()
            decomp_total.to_csv(out_dir / 'profit_decomposition_period_total.csv', index=False)
        print(f"Updated available-case CSVs for {label} using {sorted(available_cases)}. Skipping only Retail-only/Both delta figures; formal review figures are exported below to Paper_Figures.")
        return period_summary, pd.DataFrame()
    period_summary, period_delta = _make_period_summary(period_df, out_dir)

    # 1) Daily total value violin: distribution over the selected period.
    fig, ax = plt.subplots(figsize=(7.6, 4.2), constrained_layout=True)
    positions, data, colors, labels = [], [], [], []
    x = 1.0
    for scenario in SCENARIO_ORDER:
        if scenario not in set(period_df["Scenario"]):
            continue
        for case in CASE_ORDER:
            vals = period_df[(period_df["Scenario"] == scenario) & (period_df["Case"] == case)]["Total Revenue"].dropna().values
            if len(vals) == 0:
                continue
            positions.append(x)
            data.append(vals)
            colors.append(CASE_COLORS[case])
            labels.append(scenario.replace("-", "\n") + "\n" + case)
            x += 0.82
        x += 0.45
    if data:
        parts = ax.violinplot(data, positions=positions, widths=0.58, showmedians=True, showextrema=False)
        for body, color in zip(parts["bodies"], colors):
            body.set_facecolor(color)
            body.set_edgecolor("#222222")
            body.set_alpha(0.28)
            body.set_linewidth(0.8)
        parts["cmedians"].set_color("#222222")
        parts["cmedians"].set_linewidth(1.1)
        rng = np.random.default_rng(7)
        for pos, vals, color in zip(positions, data, colors):
            jitter = rng.normal(0, 0.055, size=len(vals))
            ax.scatter(np.full(len(vals), pos) + jitter, vals, s=13, color=color, edgecolor="white", linewidth=0.35, alpha=0.78, zorder=3)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_ylabel("Daily net financial value (US$)")
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
    ax.grid(axis="y", color="#bdbdbd", alpha=0.32, linewidth=0.7)
    ax.legend(handles=[Patch(facecolor=CASE_COLORS[c], edgecolor="none", label=c, alpha=0.75) for c in CASE_ORDER], frameon=False, loc="upper left")
    ax.set_title(f"Daily financial value distribution ({title_suffix})", fontweight="bold")
    _save_period_fig(fig, out_dir, "fig_financial_violin_daily_total.png")
    plt.close(fig)

    # 2) Heatmap: incremental Both minus Retail value by component.
    heat_metrics = ["Total Revenue", "WM Revenue", "TOU Cost", "PD Cost", "NCD Cost"]
    heat = period_delta.groupby("Scenario")[heat_metrics].sum(numeric_only=True).reindex(SCENARIO_ORDER)
    fig, ax = plt.subplots(figsize=(7.0, 3.35), constrained_layout=True)
    max_abs = float(np.nanmax(np.abs(heat.values))) if heat.size and np.isfinite(heat.values).any() else 1.0
    if max_abs == 0:
        max_abs = 1.0
    im = ax.imshow(heat.values, cmap="RdBu", vmin=-max_abs, vmax=max_abs, aspect="auto")
    ax.set_xticks(np.arange(len(heat_metrics)))
    ax.set_xticklabels([m.replace(" Revenue", "\nRevenue").replace(" Cost", "\nCost") for m in heat_metrics])
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels([s.replace("-", "\n") for s in heat.index])
    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            val = heat.iloc[i, j]
            text = f"{val/1000:.1f}k" if abs(val) >= 1000 else f"{val:.0f}"
            ax.text(j, i, text, ha="center", va="center", fontsize=8, color="#111111")
    ax.set_title(f"Wholesale-enabled incremental value ({title_suffix})", fontweight="bold")
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.025)
    cbar.ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
    cbar.set_label("Both minus retail-only (US$)")
    _save_period_fig(fig, out_dir, "fig_financial_delta_heatmap.png")
    plt.close(fig)

    # 3) Daily Retail-only vs Both comparison in separate scenario panels.
    fig, axes = plt.subplots(2, 2, figsize=(10.2, 5.8), sharex=False, sharey=True, constrained_layout=True)
    axes = axes.flatten()
    bar_w = 0.38
    for ax, scenario in zip(axes, SCENARIO_ORDER):
        sub = period_df[period_df["Scenario"] == scenario].copy()
        if sub.empty:
            ax.set_visible(False)
            continue
        dates = sorted(sub["Date"].unique())
        x = np.arange(len(dates))
        retail = sub[sub["Case"] == "Retail only"].set_index("Date").reindex(dates)["Total Revenue"].values
        both = sub[sub["Case"] == "Both"].set_index("Date").reindex(dates)["Total Revenue"].values
        delta = both - retail
        ax.bar(x - bar_w/2, retail, width=bar_w, color=CASE_COLORS["Retail only"], alpha=0.78, label="Retail only")
        ax.bar(x + bar_w/2, both, width=bar_w, color=CASE_COLORS["Both"], alpha=0.78, label="Both")
        ax.plot(x, delta, color="#111111", marker="o", markersize=3.2, linewidth=1.0, label="Both - Retail")
        ax.axhline(0, color="#333333", linewidth=0.8)
        ax.set_title(scenario.replace("-", " + "), fontweight="bold", pad=5)
        tick_step = max(1, int(np.ceil(len(dates) / 10)))
        ax.set_xticks(x[::tick_step])
        ax.set_xticklabels([pd.Timestamp(d).strftime("%m/%d") for d in dates[::tick_step]], rotation=0)
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
        ax.grid(axis="y", color="#bdbdbd", alpha=0.30, linewidth=0.7)
        if len(delta):
            worst_idx = np.argsort(delta)[:min(2, len(delta))]
            for idx in worst_idx:
                if delta[idx] < 0:
                    ax.annotate(f"{delta[idx]:.0f}", (x[idx], delta[idx]), textcoords="offset points", xytext=(0, -12), ha="center", fontsize=7.5, color="#111111")
    axes[0].set_ylabel("Daily value / delta (US$)")
    axes[2].set_ylabel("Daily value / delta (US$)")
    handles = [
        Patch(facecolor=CASE_COLORS["Retail only"], label="Retail only", alpha=0.78),
        Patch(facecolor=CASE_COLORS["Both"], label="Both", alpha=0.78),
        plt.Line2D([0], [0], color="#111111", marker="o", linewidth=1.0, label="Both - Retail"),
    ]
    fig.legend(handles=handles, ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.03))
    fig.suptitle(f"Retail-only and wholesale-enabled daily values ({title_suffix})", fontsize=12, fontweight="bold", y=1.08)
    _save_period_fig(fig, out_dir, "fig_financial_case_comparison_subplots.png")
    plt.close(fig)

    # 4) Value stack for Retail-only and Both in each scenario.
    components = ["EV Revenue", "WM Revenue", "TOU Cost", "PD Cost", "NCD Cost"]
    fig, axes = plt.subplots(2, 2, figsize=(10.2, 5.8), sharey=True, constrained_layout=True)
    axes = axes.flatten()
    for ax, scenario in zip(axes, SCENARIO_ORDER):
        scen_sum = period_summary[period_summary["Scenario"] == scenario].set_index("Case").reindex(CASE_ORDER)
        xs = np.arange(len(CASE_ORDER))
        pos_bottom = np.zeros(len(CASE_ORDER))
        neg_bottom = np.zeros(len(CASE_ORDER))
        for comp in components:
            vals = scen_sum[comp].fillna(0).values
            bottoms = np.where(vals >= 0, pos_bottom, neg_bottom)
            ax.bar(xs, vals, bottom=bottoms, width=0.58, color=COMPONENT_COLORS[comp], edgecolor="white", linewidth=0.55, label=comp)
            pos_bottom += np.where(vals >= 0, vals, 0)
            neg_bottom += np.where(vals < 0, vals, 0)
        net = scen_sum["Total Revenue"].fillna(0).values
        ax.plot(xs, net, color="#111111", marker="o", markersize=4.0, linewidth=1.1)
        for xi, yi in zip(xs, net):
            ax.annotate(f"${yi:,.0f}", (xi, yi), textcoords="offset points", xytext=(0, 6 if yi >= 0 else -12), ha="center", fontsize=7.5)
        ax.axhline(0, color="#333333", linewidth=0.8)
        ax.set_title(scenario.replace("-", " + "), fontweight="bold", pad=5)
        ax.set_xticks(xs)
        ax.set_xticklabels(CASE_ORDER)
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
        ax.grid(axis="y", color="#bdbdbd", alpha=0.30, linewidth=0.7)
    axes[0].set_ylabel("Period value (US$)")
    axes[2].set_ylabel("Period value (US$)")
    handles = [Patch(facecolor=COMPONENT_COLORS[c], edgecolor="none", label=c) for c in components]
    fig.legend(handles=handles, ncol=5, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.03))
    fig.suptitle(f"Retail-only and wholesale-enabled value stacks ({title_suffix})", fontsize=12, fontweight="bold", y=1.08)
    _save_period_fig(fig, out_dir, "fig_financial_case_value_stack_subplots.png")
    plt.close(fig)

    # 5) Daily profit decomposition: WM/NWM x EV/BESS x energy/capacity.
    missing_decomp = [col for col in DECOMP_COMPONENTS if col not in period_df.columns]
    if missing_decomp:
        print(f"Skipping profit decomposition for {label}; rerun both MPC notebooks to create: {missing_decomp}")
    else:
        both_decomp = period_df[period_df['Case'] == 'Both'][['Scenario', 'Date', 'Total Revenue'] + DECOMP_COMPONENTS].copy()
        both_decomp = both_decomp.sort_values(['Scenario', 'Date'])
        both_decomp['Decomposition Residual'] = both_decomp['Total Revenue'] - both_decomp[DECOMP_COMPONENTS].sum(axis=1)
        both_decomp.to_csv(out_dir / 'profit_decomposition_daily.csv', index=False)
        decomp_total = both_decomp.groupby('Scenario', as_index=False)[['Total Revenue'] + DECOMP_COMPONENTS + ['Decomposition Residual']].sum(numeric_only=True)
        decomp_total = decomp_total.set_index('Scenario').reindex(SCENARIO_ORDER).dropna(how='all').reset_index()
        decomp_total.to_csv(out_dir / 'profit_decomposition_period_total.csv', index=False)

        group_handles = [Patch(facecolor=color, edgecolor='none', label=group) for group, color in DECOMP_GROUP_COLORS.items()]
        type_handles = [
            Patch(facecolor='#888888', edgecolor='#333333', label='Energy'),
            Patch(facecolor='#888888', edgecolor='#333333', hatch='////', label='Capacity'),
        ]
        decomp_handles = group_handles + type_handles
        decomp_ncol = max(2, min(len(decomp_handles), int(10.2 // 2.0)))

        fig, axes = plt.subplots(2, 2, figsize=(10.2, 6.4), sharey=True, constrained_layout=False)
        fig.subplots_adjust(top=0.78, hspace=0.30, wspace=0.08)
        axes = axes.flatten()
        for ax, scenario in zip(axes, SCENARIO_ORDER):
            sub = both_decomp[both_decomp['Scenario'] == scenario].sort_values('Date')
            if sub.empty:
                ax.set_visible(False)
                continue
            x = np.arange(len(sub))
            pos_bottom = np.zeros(len(sub))
            neg_bottom = np.zeros(len(sub))
            for component in DECOMP_COMPONENTS:
                values = sub[component].fillna(0).to_numpy()
                bottoms = np.where(values >= 0, pos_bottom, neg_bottom)
                group = component.rsplit(' ', 1)[0]
                hatch = '////' if component.endswith('Capacity') else None
                ax.bar(x, values, bottom=bottoms, width=0.76, color=DECOMP_GROUP_COLORS[group], hatch=hatch, edgecolor='white', linewidth=0.35)
                pos_bottom += np.where(values >= 0, values, 0)
                neg_bottom += np.where(values < 0, values, 0)
            ax.plot(x, sub['Total Revenue'], color='#111111', marker='o', markersize=2.8, linewidth=0.9, label='Net total')
            ax.axhline(0, color='#333333', linewidth=0.8)
            tick_step = max(1, int(np.ceil(len(sub) / 10)))
            ax.set_xticks(x[::tick_step])
            ax.set_xticklabels([pd.Timestamp(d).strftime('%m/%d') for d in sub['Date'].iloc[::tick_step]])
            ax.set_title(scenario.replace('-', ' + '), fontweight='bold', pad=5)
            ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
            ax.grid(axis='y', color='#bdbdbd', alpha=0.30, linewidth=0.7)
        axes[0].set_ylabel('Daily profit contribution (US$)')
        axes[2].set_ylabel('Daily profit contribution (US$)')
        fig.legend(handles=decomp_handles, ncol=decomp_ncol, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 0.925))
        fig.suptitle(f'Daily profit decomposition ({title_suffix})', fontsize=12, fontweight='bold', y=0.985)
        _save_period_fig(fig, out_dir, 'fig_profit_decomposition_daily.png')
        plt.close(fig)

        fig, ax = plt.subplots(figsize=(8.5, 4.6), constrained_layout=True)
        x = np.arange(len(decomp_total))
        pos_bottom = np.zeros(len(decomp_total))
        neg_bottom = np.zeros(len(decomp_total))
        for component in DECOMP_COMPONENTS:
            values = decomp_total[component].fillna(0).to_numpy()
            bottoms = np.where(values >= 0, pos_bottom, neg_bottom)
            group = component.rsplit(' ', 1)[0]
            hatch = '////' if component.endswith('Capacity') else None
            ax.bar(x, values, bottom=bottoms, width=0.62, color=DECOMP_GROUP_COLORS[group], hatch=hatch, edgecolor='white', linewidth=0.45)
            pos_bottom += np.where(values >= 0, values, 0)
            neg_bottom += np.where(values < 0, values, 0)
        net_values = decomp_total['Total Revenue'].to_numpy()
        ax.plot(x, net_values, color='#111111', marker='D', linestyle='none', markersize=4.6, label='Net total')
        for xi, yi in zip(x, net_values):
            ax.annotate(f'${yi:,.0f}', (xi, yi), textcoords='offset points', xytext=(0, 6 if yi >= 0 else -12), ha='center', fontsize=8)
        ax.axhline(0, color='#333333', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels([s.replace('-', '\n') for s in decomp_total['Scenario']])
        ax.set_ylabel('Period profit contribution (US$)')
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
        ax.grid(axis='y', color='#bdbdbd', alpha=0.30, linewidth=0.7)
        ax.legend(handles=decomp_handles, ncol=max(2, min(len(decomp_handles), int(8.5 // 2.0))), frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.16))
        ax.set_title(f'Period-total profit decomposition ({title_suffix})', fontweight='bold', pad=36)
        _save_period_fig(fig, out_dir, 'fig_profit_decomposition_period_total.png')
        plt.close(fig)

    return period_summary, period_delta


period_specs = []
if not all_df.empty:
    months = sorted(all_df["Date"].dt.to_period("M").unique())
    if len(months) > 1:
        full_label = "combined_" + _period_label(all_df)
        full_title = f"{pd.Timestamp(all_df['Date'].min()).strftime('%b %d')}--{pd.Timestamp(all_df['Date'].max()).strftime('%b %d, %Y')}"
        period_specs.append((full_label, all_df.copy(), full_title))

    for month in months:
        month_df = all_df[all_df["Date"].dt.to_period("M") == month].copy()
        period_specs.append((str(month), month_df, pd.Period(month).strftime("%B %Y")))

period_outputs = []
for label, period_df, title_suffix in period_specs:
    print(f"\nGenerating period figures: {label} ({len(period_df)} case-day rows)")
    period_summary, period_delta = plot_period_financials(period_df, label, title_suffix)
    period_outputs.append((label, len(period_df)))

print("\nGenerated period outputs:")
for label, n in period_outputs:
    print(f" - {label}: {n} case-day rows")




Generating period figures: 2025-06 (240 case-day rows)
Paper_Figures/financial_comparison/2025-06/fig_financial_violin_daily_total.png


Paper_Figures/financial_comparison/2025-06/fig_financial_delta_heatmap.png


Paper_Figures/financial_comparison/2025-06/fig_financial_case_comparison_subplots.png
Paper_Figures/financial_comparison/2025-06/fig_financial_case_value_stack_subplots.png


Paper_Figures/financial_comparison/2025-06/fig_profit_decomposition_daily.png
Paper_Figures/financial_comparison/2025-06/fig_profit_decomposition_period_total.png

Generated period outputs:
 - 2025-06: 240 case-day rows


## SEGAN Version 2.7: formal June 2025 paper figures

This section is independent of the legacy retail-only/Both 2 x 2 figures above. It reads the versioned formal Rolling/oracle snapshot and the latest complete Shrinking outputs, then writes journal-facing review figures to `Paper_Figures`. Only after validation should these files be copied to `Latex/paper2.7`.


In [4]:
# Formal June 2025 exports for the SEGAN v2.7 manuscript.
# Keep this code in the source plotting notebook; do not create a standalone plotting script.
FORMAL_REVIEW_DIR = PROJECT / 'Paper_Figures' / 'financial_comparison' / '2025-06'
FORMAL_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

FORMAL_ROOT = PROJECT / 'Validation_Results/June_2025/formal_runs/gate_e_20260726_c8250f11'
FORMAL_DAILY_SOURCES = [
    ('Shrinking 24 h Perfect', PROJECT / 'Results_Shrinking/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv'),
    ('Shrinking 24 h Persistence', PROJECT / 'Results_Shrinking/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv'),
    ('Rolling 24 h Perfect', FORMAL_ROOT / 'rolling/Plots/Cost/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival/daily_financial_detail.csv'),
    ('Rolling 24 h Persistence', FORMAL_ROOT / 'rolling/Plots/Cost/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival/daily_financial_detail.csv'),
]
ORACLE_DAILY_PATH = FORMAL_ROOT / 'rolling_perfect/oracle/oracle_daily_financial.csv'

required_paths = [path for _, path in FORMAL_DAILY_SOURCES] + [ORACLE_DAILY_PATH]
missing_formal = [path for path in required_paths if not path.exists()]
if missing_formal:
    raise FileNotFoundError('Missing formal paper inputs:\n' + '\n'.join(str(path) for path in missing_formal))

formal_frames = []
decomp_frames = []
for method, path in FORMAL_DAILY_SOURCES:
    frame = pd.read_csv(path)
    frame['Date'] = pd.to_datetime(frame['Date'])
    frame = frame[frame['Case'].eq('Both')].copy()
    if len(frame) != 30 or frame['Date'].nunique() != 30:
        raise ValueError(f'{method}: expected 30 unique June days, found {len(frame)} rows and {frame["Date"].nunique()} dates')
    frame['Method'] = method
    formal_frames.append(frame)
    decomp_frames.append(frame[['Method', 'Date', 'Total Revenue'] + DECOMP_COMPONENTS].copy())

oracle_daily = pd.read_csv(ORACLE_DAILY_PATH)
oracle_daily['Date'] = pd.to_datetime(oracle_daily['Date'])
if len(oracle_daily) != 30 or oracle_daily['Date'].nunique() != 30:
    raise ValueError('Oracle daily file does not contain exactly 30 unique June days')
oracle_daily['Method'] = 'Full billing-period Perfect oracle'
formal_frames.append(oracle_daily)

formal_daily = pd.concat(formal_frames, ignore_index=True, sort=False)
formal_daily = formal_daily.sort_values(['Method', 'Date'])
formal_daily.to_csv(FORMAL_REVIEW_DIR / 'formal_daily_financial.csv', index=False)

paper_method_order = [
    'Shrinking 24 h Perfect',
    'Shrinking 24 h Persistence',
    'Rolling 24 h Perfect',
    'Rolling 24 h Persistence',
    'Full billing-period Perfect oracle',
]
paper_method_labels = {
    'Shrinking 24 h Perfect': 'Shrinking\nPerfect',
    'Shrinking 24 h Persistence': 'Shrinking\nPersistence',
    'Rolling 24 h Perfect': 'Rolling\nPerfect',
    'Rolling 24 h Persistence': 'Rolling\nPersistence',
    'Full billing-period Perfect oracle': 'Full-period\nPerfect oracle',
}
paper_components = ['EV Revenue', 'WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost']
formal_monthly = formal_daily.groupby('Method', as_index=False)[METRICS].sum(numeric_only=True)
formal_monthly = formal_monthly.set_index('Method').reindex(paper_method_order).reset_index()
formal_monthly.to_csv(FORMAL_REVIEW_DIR / 'formal_monthly_financial.csv', index=False)

def save_formal_review(fig, stem):
    fig.savefig(FORMAL_REVIEW_DIR / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)

# Figure 1: complete monthly net-value accounting for the five formal methods.
fig, ax = plt.subplots(figsize=(8.7, 4.7), constrained_layout=True)
y = np.arange(len(formal_monthly))
pos_left = np.zeros(len(formal_monthly))
neg_left = np.zeros(len(formal_monthly))
for component in paper_components:
    values = formal_monthly[component].to_numpy(float)
    left = np.where(values >= 0, pos_left, neg_left)
    ax.barh(y, values, left=left, height=0.60, color=COMPONENT_COLORS[component], edgecolor='white', linewidth=0.55, label=component)
    pos_left += np.where(values >= 0, values, 0)
    neg_left += np.where(values < 0, values, 0)
net_values = formal_monthly['Total Revenue'].to_numpy(float)
ax.scatter(net_values, y, color='#111111', marker='D', s=26, zorder=4, label='Net value')
for yi, value in zip(y, net_values):
    ax.annotate(f'${value:,.0f}', (value, yi), xytext=(6, 0), textcoords='offset points', va='center', fontsize=8)
ax.axvline(0, color='#333333', linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels([paper_method_labels[m] for m in formal_monthly['Method']])
ax.invert_yaxis()
ax.set_xlabel('June 2025 value (US$)')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(dollars))
ax.grid(axis='x', color='#bdbdbd', alpha=0.32, linewidth=0.7)
ax.legend(ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.19))
ax.set_title('Executed monthly net value and financial components', fontweight='bold', pad=42)
save_formal_review(fig, 'fig_01_monthly_five_method_value_stack')

# Figure 2: daily and cumulative value for the controller emphasized in the paper.
focus_methods = ['Rolling 24 h Persistence', 'Rolling 24 h Perfect', 'Full billing-period Perfect oracle']
focus_colors = {
    'Rolling 24 h Persistence': '#E69F00',
    'Rolling 24 h Perfect': '#0072B2',
    'Full billing-period Perfect oracle': '#009E73',
}
fig, axes = plt.subplots(2, 1, figsize=(8.7, 5.7), sharex=True, constrained_layout=True)
for method in focus_methods:
    sub = formal_daily[formal_daily['Method'].eq(method)].sort_values('Date')
    axes[0].plot(sub['Date'], sub['Total Revenue'], color=focus_colors[method], linewidth=1.25, marker='o', markersize=2.7, label=method)
    axes[1].plot(sub['Date'], sub['Total Revenue'].cumsum(), color=focus_colors[method], linewidth=1.6, label=method)
axes[0].axhline(0, color='#333333', linewidth=0.8)
axes[0].set_ylabel('Daily net value (US$)')
axes[1].set_ylabel('Cumulative net value (US$)')
axes[1].set_xlabel('Date in June 2025')
for ax in axes:
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
    ax.grid(color='#bdbdbd', alpha=0.30, linewidth=0.7)
axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=4))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[1].set_xlim(formal_daily['Date'].min(), formal_daily['Date'].max())
axes[0].legend(ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.27))
axes[0].set_title('Daily and cumulative executed value', fontweight='bold', pad=39)
save_formal_review(fig, 'fig_02_daily_rolling_oracle_value')

# Figure 3: four-policy WM/NWM, EV/BESS, energy/capacity attribution.
formal_decomp = pd.concat(decomp_frames, ignore_index=True)
formal_decomp.to_csv(FORMAL_REVIEW_DIR / 'formal_daily_profit_decomposition.csv', index=False)
decomp_total = formal_decomp.groupby('Method', as_index=False)[['Total Revenue'] + DECOMP_COMPONENTS].sum(numeric_only=True)
decomp_total = decomp_total.set_index('Method').reindex(paper_method_order[:-1]).reset_index()
decomp_total['Decomposition Residual'] = decomp_total['Total Revenue'] - decomp_total[DECOMP_COMPONENTS].sum(axis=1)
decomp_total.to_csv(FORMAL_REVIEW_DIR / 'formal_monthly_profit_decomposition.csv', index=False)
fig, ax = plt.subplots(figsize=(8.7, 4.8), constrained_layout=True)
x = np.arange(len(decomp_total))
pos_bottom = np.zeros(len(decomp_total))
neg_bottom = np.zeros(len(decomp_total))
for component in DECOMP_COMPONENTS:
    values = decomp_total[component].to_numpy(float)
    bottom = np.where(values >= 0, pos_bottom, neg_bottom)
    group = component.rsplit(' ', 1)[0]
    hatch = '////' if component.endswith('Capacity') else None
    ax.bar(x, values, bottom=bottom, width=0.62, color=DECOMP_GROUP_COLORS[group], hatch=hatch, edgecolor='white', linewidth=0.45)
    pos_bottom += np.where(values >= 0, values, 0)
    neg_bottom += np.where(values < 0, values, 0)
ax.scatter(x, decomp_total['Total Revenue'], color='#111111', marker='D', s=28, zorder=4)
ax.axhline(0, color='#333333', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([paper_method_labels[m] for m in decomp_total['Method']])
ax.set_ylabel('June 2025 profit contribution (US$)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
ax.grid(axis='y', color='#bdbdbd', alpha=0.30, linewidth=0.7)
legend_handles = [Patch(facecolor=color, edgecolor='none', label=group) for group, color in DECOMP_GROUP_COLORS.items()]
legend_handles += [Patch(facecolor='#888888', edgecolor='#333333', label='Energy'), Patch(facecolor='#888888', edgecolor='#333333', hatch='////', label='Capacity')]
ax.legend(handles=legend_handles, ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.22))
ax.set_title('Executed profit attribution by resource and service channel', fontweight='bold', pad=82)
save_formal_review(fig, 'fig_03_monthly_resource_service_decomposition')

# Figure 4: daily attribution retained as a paper appendix/supplement candidate.
fig, axes = plt.subplots(2, 2, figsize=(10.2, 6.4), sharey=True, constrained_layout=False)
fig.subplots_adjust(top=0.80, hspace=0.30, wspace=0.10)
for ax, method in zip(axes.flat, paper_method_order[:-1]):
    sub = formal_decomp[formal_decomp['Method'].eq(method)].sort_values('Date')
    xday = np.arange(len(sub))
    pos_bottom = np.zeros(len(sub))
    neg_bottom = np.zeros(len(sub))
    for component in DECOMP_COMPONENTS:
        values = sub[component].to_numpy(float)
        bottom = np.where(values >= 0, pos_bottom, neg_bottom)
        group = component.rsplit(' ', 1)[0]
        hatch = '////' if component.endswith('Capacity') else None
        ax.bar(xday, values, bottom=bottom, width=0.78, color=DECOMP_GROUP_COLORS[group], hatch=hatch, edgecolor='white', linewidth=0.25)
        pos_bottom += np.where(values >= 0, values, 0)
        neg_bottom += np.where(values < 0, values, 0)
    ax.plot(xday, sub['Total Revenue'], color='#111111', linewidth=0.9, marker='o', markersize=2.4)
    ax.axhline(0, color='#333333', linewidth=0.8)
    ax.set_title(method.replace(' 24 h ', ' + '), fontweight='bold')
    ax.set_xticks(xday[::4])
    ax.set_xticklabels([pd.Timestamp(d).strftime('%m/%d') for d in sub['Date'].iloc[::4]])
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars))
    ax.grid(axis='y', color='#bdbdbd', alpha=0.30, linewidth=0.7)
axes[0, 0].set_ylabel('Daily profit contribution (US$)')
axes[1, 0].set_ylabel('Daily profit contribution (US$)')
fig.legend(handles=legend_handles, ncol=6, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 0.92))
fig.suptitle('Daily executed profit attribution', fontsize=12, fontweight='bold', y=0.985)
save_formal_review(fig, 'fig_04_daily_resource_service_decomposition')

print('SEGAN v2.7 formal review figures written to:', FORMAL_REVIEW_DIR.relative_to(PROJECT))
print(formal_monthly[['Method', 'Total Revenue']].to_string(index=False))


SEGAN v2.7 formal review figures written to: Paper_Figures/financial_comparison/2025-06
                            Method  Total Revenue
            Shrinking 24 h Perfect   10549.330000
        Shrinking 24 h Persistence   11131.920000
              Rolling 24 h Perfect   10733.150000
          Rolling 24 h Persistence   10485.280000
Full billing-period Perfect oracle   12565.154573


## SEGAN Version 2.8: Rolling-first paper and daily-validation figures

This section is the single source for all v2.8 paper figures. It reads only the frozen Gate E snapshot, generates 300-dpi PNG files, keeps all 30 daily validation figures in `Paper_Figures`, and copies only manuscript-selected figures to `Latex/paper2.8`. The bid panels retain separate DA and RT bars; the obligation panels plot the physically executed net obligation and the original physical bounds.

In [5]:
# Rolling-first formal exports for the SEGAN v2.8 manuscript.
import hashlib
import shutil

GATE_E_TAG = 'gate_e_20260726_c8250f11'
GATE_E_ROOT = PROJECT / 'Validation_Results/June_2025/formal_runs' / GATE_E_TAG
PAPER_V28_ROOT = PROJECT / 'Paper_Figures/financial_comparison/2025-06/paper_v2.8'
PAPER_V28_DAILY = PAPER_V28_ROOT / 'daily_validation'
LATEX_V28_FIGURES = PROJECT / 'Latex/paper2.8'
for directory in [PAPER_V28_ROOT, PAPER_V28_DAILY, LATEX_V28_FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)

V28_METHODS = ['Rolling 24 h Persistence', 'Rolling 24 h Perfect', 'Full billing-period Perfect oracle']
V28_LABELS = {
    'Rolling 24 h Persistence': 'Rolling 24 h\nPersistence',
    'Rolling 24 h Perfect': 'Rolling 24 h\nPerfect',
    'Full billing-period Perfect oracle': 'Full-period\nPerfect oracle',
}
V28_COLORS = {
    'Rolling 24 h Persistence': '#E69F00',
    'Rolling 24 h Perfect': '#0072B2',
    'Full billing-period Perfect oracle': '#009E73',
}
V28_COMPONENTS = ['EV Revenue', 'WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost']
V28_SELECTED_DAYS = [pd.Timestamp('2025-06-03')]
stale_paper_figure = LATEX_V28_FIGURES / 'fig_06_daily_validation_2025-06-13.png'
if stale_paper_figure.exists():
    stale_paper_figure.unlink()

monthly_path = GATE_E_ROOT / 'gate_e_monthly_financial_comparison.csv'
daily_path = GATE_E_ROOT / 'gate_e_daily_financial_comparison.csv'
oracle_dispatch_path = GATE_E_ROOT / 'rolling_perfect/oracle/oracle_dispatch.csv'
required_v28 = [monthly_path, daily_path, oracle_dispatch_path]
missing_v28 = [p for p in required_v28 if not p.exists()]
if missing_v28:
    raise FileNotFoundError('Missing Gate E input(s):\n' + '\n'.join(str(p) for p in missing_v28))

v28_monthly = pd.read_csv(monthly_path).rename(columns={'Scenario': 'Method'})
v28_daily = pd.read_csv(daily_path).rename(columns={'Scenario': 'Method'})
v28_daily['Date'] = pd.to_datetime(v28_daily['Date'])
v28_monthly = v28_monthly.set_index('Method').reindex(V28_METHODS).reset_index()
if len(v28_monthly) != 3 or v28_monthly[V28_COMPONENTS + ['Total Revenue']].isna().any().any():
    raise ValueError('Gate E monthly table is incomplete for the three Rolling-first methods')
for method in V28_METHODS:
    sub = v28_daily[v28_daily['Method'].eq(method)]
    if len(sub) != 30 or sub['Date'].nunique() != 30:
        raise ValueError(f'{method}: expected 30 unique June days')

def _sha256_v28(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

v28_manifest_rows = []
# Reuse reviewed static and executed-daily figures without redrawing them.
# The two legacy graphics supplied by the author are stored under
# Paper_Figures/static_sources so this notebook remains reproducible.
static_v28 = [
    (PROJECT / 'Latex/paper2.7/fig_00_system_architecture.png', 'fig_00_system_architecture.png'),
    (PROJECT / 'Latex/paper2.7/fig_00_market_mpc_timeline.png', 'fig_00_market_mpc_timeline.png'),
    (PROJECT / 'Paper_Figures/static_sources/legacy_market_bidding_mpc_timeline.png', 'fig_06_legacy_market_bidding_mpc_timeline.png'),
    (PROJECT / 'Paper_Figures/static_sources/legacy_market_interface_context.png', 'fig_07_legacy_market_interface_context.png'),
    (PROJECT / 'Results_Rolling/Plots/Solver_RT_Choices/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival_full/20250603/RT_6panel_2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival_full.png', 'fig_08_rolling_persistence_daily_6panel_2025-06-03.png'),
    (PROJECT / 'Results_Rolling/Plots/Solver_RT_Choices/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival_full/20250603/RT_6panel_2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival_full.png', 'fig_09_rolling_perfect_daily_6panel_2025-06-03.png'),
]
for static_source, static_name in static_v28:
    if not static_source.exists(): raise FileNotFoundError(static_source)
    shutil.copy2(static_source, LATEX_V28_FIGURES / static_name)
    shutil.copy2(static_source, PAPER_V28_ROOT / static_name)
    v28_manifest_rows.append({'figure': static_name, 'formal_tag': GATE_E_TAG, 'paper_copy': True, 'source_files': str(static_source.relative_to(PROJECT)), 'source_sha256': _sha256_v28(static_source)})

def save_v28(fig, stem, manuscript=True, source_paths=()):
    review_path = PAPER_V28_ROOT / f'{stem}.png'
    fig.savefig(review_path, dpi=300, bbox_inches='tight', facecolor='white')
    if manuscript:
        fig.savefig(LATEX_V28_FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    v28_manifest_rows.append({
        'figure': f'{stem}.png', 'formal_tag': GATE_E_TAG,
        'paper_copy': bool(manuscript),
        'source_files': ';'.join(str(Path(p).relative_to(PROJECT)) for p in source_paths),
        'source_sha256': ';'.join(_sha256_v28(p) for p in source_paths),
    })
    return review_path

# Figure 1: complete financial identity for the three formal methods.
fig, ax = plt.subplots(figsize=(8.7, 4.0), constrained_layout=True)
y = np.arange(len(v28_monthly))
pos_left = np.zeros(len(v28_monthly)); neg_left = np.zeros(len(v28_monthly))
for component in V28_COMPONENTS:
    values = v28_monthly[component].to_numpy(float)
    left = np.where(values >= 0, pos_left, neg_left)
    ax.barh(y, values, left=left, height=0.58, color=COMPONENT_COLORS[component], edgecolor='white', linewidth=0.55, label=component)
    pos_left += np.where(values >= 0, values, 0); neg_left += np.where(values < 0, values, 0)
net = v28_monthly['Total Revenue'].to_numpy(float)
ax.scatter(net, y, color='#111111', marker='D', s=30, zorder=4, label='Net value')
for yi, value in zip(y, net):
    ax.annotate(f'${value:,.0f}', (value, yi), xytext=(6, 0), textcoords='offset points', va='center', fontsize=8.5)
ax.axvline(0, color='#333333', linewidth=0.8)
ax.set_yticks(y); ax.set_yticklabels([V28_LABELS[m] for m in v28_monthly['Method']]); ax.invert_yaxis()
ax.set_xlabel('June 2025 executed value (US$)'); ax.xaxis.set_major_formatter(ticker.FuncFormatter(dollars))
ax.grid(axis='x', color='#bdbdbd', alpha=0.32, linewidth=0.7)
ax.legend(ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.23))
ax.set_title('Forecast value and billing-horizon value under Rolling control', fontweight='bold', pad=46)
save_v28(fig, 'fig_01_monthly_rolling_oracle_value_stack', source_paths=[monthly_path])

# Figure 2: daily attribution and cumulative billing-period value.
fig, axes = plt.subplots(2, 1, figsize=(8.7, 5.7), sharex=True, constrained_layout=True)
for method in V28_METHODS:
    sub = v28_daily[v28_daily['Method'].eq(method)].sort_values('Date')
    axes[0].plot(sub['Date'], sub['Total Revenue'], color=V28_COLORS[method], lw=1.25, marker='o', ms=2.6, label=method)
    axes[1].plot(sub['Date'], sub['Total Revenue'].cumsum(), color=V28_COLORS[method], lw=1.65, label=method)
axes[0].axhline(0, color='#333333', lw=0.8); axes[0].set_ylabel('Daily attributed value (US$)')
axes[1].set_ylabel('Cumulative value (US$)'); axes[1].set_xlabel('Date in June 2025')
for ax in axes:
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars)); ax.grid(color='#bdbdbd', alpha=0.30, lw=0.7)
axes[1].xaxis.set_major_locator(mdates.DayLocator(interval=4)); axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
axes[0].legend(ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.28))
axes[0].set_title('Executed daily and cumulative value', fontweight='bold', pad=42)
save_v28(fig, 'fig_02_daily_rolling_oracle_value', source_paths=[daily_path])

# Figure 3: separate forecast value from horizon value component by component.
monthly_idx = v28_monthly.set_index('Method')
gap_table = pd.DataFrame({
    'Forecast value: Rolling Perfect - Persistence': monthly_idx.loc['Rolling 24 h Perfect', V28_COMPONENTS] - monthly_idx.loc['Rolling 24 h Persistence', V28_COMPONENTS],
    'Horizon value: Oracle - Rolling Perfect': monthly_idx.loc['Full billing-period Perfect oracle', V28_COMPONENTS] - monthly_idx.loc['Rolling 24 h Perfect', V28_COMPONENTS],
}).T
gap_table['Net value difference'] = gap_table[V28_COMPONENTS].sum(axis=1)
gap_table.to_csv(PAPER_V28_ROOT / 'forecast_horizon_value_gap_decomposition.csv', float_format='%.6f')
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.8), sharey=True, constrained_layout=True)
for ax, (gap_name, row) in zip(axes, gap_table.iterrows()):
    vals = row[V28_COMPONENTS].to_numpy(float); x = np.arange(len(V28_COMPONENTS))
    ax.bar(x, vals, color=[COMPONENT_COLORS[c] for c in V28_COMPONENTS], edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='#333333', lw=0.8); ax.grid(axis='y', color='#bdbdbd', alpha=0.30, lw=0.7)
    ax.set_xticks(x); ax.set_xticklabels(['EV', 'WM', 'TOU', 'PD', 'NCD'])
    ax.set_title(gap_name.replace(': ', ':\n'), fontweight='bold', fontsize=9.5)
    ax.text(0.98, 0.96, f"Net = ${row['Net value difference']:,.2f}", transform=ax.transAxes, ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='#777777'))
axes[0].set_ylabel('Component contribution to value gap (US$)')
save_v28(fig, 'fig_03_forecast_horizon_gap_decomposition', source_paths=[monthly_path])

# Figure 4: Rolling-only resource/service attribution; oracle is excluded because
# the formal oracle file does not contain an EV/BESS accounting allocation.
rolling_decomp_components = ['WM EV Energy', 'WM EV Capacity', 'WM BESS Energy', 'WM BESS Capacity', 'NWM EV Energy', 'NWM EV Capacity', 'NWM BESS Energy', 'NWM BESS Capacity']
rolling_decomp = v28_daily[v28_daily['Method'].isin(V28_METHODS[:2])].groupby('Method', as_index=False)[['Total Revenue'] + rolling_decomp_components].sum(numeric_only=True)
rolling_decomp = rolling_decomp.set_index('Method').reindex(V28_METHODS[:2]).reset_index()
rolling_decomp['Decomposition Residual'] = rolling_decomp['Total Revenue'] - rolling_decomp[rolling_decomp_components].sum(axis=1)
rolling_decomp.to_csv(PAPER_V28_ROOT / 'rolling_monthly_resource_service_decomposition.csv', index=False, float_format='%.6f')
fig, ax = plt.subplots(figsize=(7.8, 4.2), constrained_layout=True)
x = np.arange(len(rolling_decomp)); pos = np.zeros(len(x)); neg = np.zeros(len(x))
for component in rolling_decomp_components:
    vals = rolling_decomp[component].to_numpy(float); bottom = np.where(vals >= 0, pos, neg)
    group = component.rsplit(' ', 1)[0]; hatch = '////' if component.endswith('Capacity') else None
    ax.bar(x, vals, bottom=bottom, width=0.58, color=DECOMP_GROUP_COLORS[group], hatch=hatch, edgecolor='white', linewidth=0.45)
    pos += np.where(vals >= 0, vals, 0); neg += np.where(vals < 0, vals, 0)
ax.scatter(x, rolling_decomp['Total Revenue'], color='#111111', marker='D', s=30, zorder=4)
ax.axhline(0, color='#333333', lw=0.8); ax.set_xticks(x); ax.set_xticklabels([V28_LABELS[m] for m in rolling_decomp['Method']])
ax.set_ylabel('June 2025 contribution (US$)'); ax.yaxis.set_major_formatter(ticker.FuncFormatter(dollars)); ax.grid(axis='y', color='#bdbdbd', alpha=0.30, lw=0.7)
handles = [Patch(facecolor=color, label=group) for group, color in DECOMP_GROUP_COLORS.items()] + [Patch(facecolor='#888888', label='Energy'), Patch(facecolor='#888888', hatch='////', label='Capacity')]
ax.legend(handles=handles, ncol=min(3, len(handles)), frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.24))
ax.set_title('Rolling executed-value attribution by resource and service', fontweight='bold', pad=50)
save_v28(fig, 'fig_04_monthly_rolling_resource_service_decomposition', source_paths=[daily_path])

# Daily validation: three information/horizon cases under a common plotting grammar.
oracle_dispatch = pd.read_csv(oracle_dispatch_path); oracle_dispatch['Interval start'] = pd.to_datetime(oracle_dispatch['Interval start'])
trace_roots = {
    'Rolling 24 h Persistence': GATE_E_ROOT / 'rolling/Validation_Traces/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival_full',
    'Rolling 24 h Perfect': GATE_E_ROOT / 'rolling/Validation_Traces/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival_full',
}
alpha = {'RU': 0.7, 'RD': 0.7, 'SP': 0.2, 'NSP': 0.2}
bid_colors = {'p_DA': '#4C78A8', 'p_RT': '#40BFC1', 'RU_DA': '#54A24B', 'RU_RT': '#9CD67D', 'SP_DA': '#B39BC8', 'SP_RT': '#D6C5E3', 'NSP_DA': '#8C6BB1', 'NSP_RT': '#C7B5D8', 'RD_DA': '#E45756', 'RD_RT': '#FF9D9A'}

def _daily_trace(method, day):
    if method == 'Full billing-period Perfect oracle':
        frame = oracle_dispatch[oracle_dispatch['Interval start'].dt.normalize().eq(day.normalize())].copy()
    else:
        path = trace_roots[method] / f'rolling_validation_trace_{day:%Y%m%d}.csv'
        if not path.exists(): raise FileNotFoundError(path)
        frame = pd.read_csv(path); frame['Interval start'] = pd.to_datetime(frame['Interval start'])
    if len(frame) != 96 or frame['Interval start'].nunique() != 96:
        raise ValueError(f'{method} {day:%Y-%m-%d}: expected 96 executed intervals')
    return frame.sort_values('Interval start').reset_index(drop=True)

def _signed_stack(ax, components, width=0.82):
    n = len(components[0][1]); x = np.arange(n); pos = np.zeros(n); neg = np.zeros(n)
    for label, values, color in components:
        values = np.asarray(values, dtype=float); bottom = np.where(values >= 0, pos, neg)
        ax.bar(x, values, bottom=bottom, width=width, color=color, edgecolor='white', linewidth=0.15, label=label)
        pos += np.where(values >= 0, values, 0); neg += np.where(values < 0, values, 0)
    return x

def _grouped_bars(ax, components, total_width=0.86):
    # Component bids are displayed side-by-side, not algebraically stacked.
    # This preserves p_DA and p_RT as separate bars without implying that
    # the positive/negative drawing stack is the physical capability test.
    n = len(components[0][1]); x = np.arange(n); width = total_width / len(components)
    offsets = (np.arange(len(components)) - (len(components)-1)/2) * width
    for offset, (label, values, color) in zip(offsets, components):
        ax.bar(x + offset, np.asarray(values, dtype=float), width=width, color=color, edgecolor='none', label=label)
    return x

def plot_v28_daily_validation(day):
    frames = {method: _daily_trace(method, day) for method in V28_METHODS}
    fig, axes = plt.subplots(5, 3, figsize=(13.2, 10.5), sharex='col', constrained_layout=False)
    fig.subplots_adjust(top=0.91, bottom=0.07, left=0.07, right=0.99, hspace=0.27, wspace=0.18)
    for col, method in enumerate(V28_METHODS):
        df = frames[method]; x = np.arange(96)
        ax = axes[0, col]
        ax.plot(x, df['p_GI_kW'], color='#111111', lw=1.1, label='Meter $p^{GI}$')
        ax.plot(x, df['p_EV_kW'], color='#E69F00', lw=0.95, label='EV')
        ax.plot(x, df['p_BESS_kW'], color='#0072B2', lw=0.95, label='BESS')
        ax.axhline(0, color='#555555', lw=0.55); ax.set_title(V28_LABELS[method].replace('\n', ' — '), fontweight='bold')
        ax = axes[1, col]
        bid_components = [
            ('$p^{DA}$', df['p_DA_kW'], bid_colors['p_DA']), ('$p^{RT}$', df['p_RT_deviation_kW'], bid_colors['p_RT']),
            ('$c^{RU,DA}$', df['c_RU_DA_kW'], bid_colors['RU_DA']), ('$c^{RU,RT}$', df['c_RU_RT_kW'], bid_colors['RU_RT']),
            ('$c^{SP,DA}$', df['c_SP_DA_kW'], bid_colors['SP_DA']), ('$c^{SP,RT}$', df['c_SP_RT_kW'], bid_colors['SP_RT']),
            ('$c^{NSP,DA}$', df['c_NSP_DA_kW'], bid_colors['NSP_DA']), ('$c^{NSP,RT}$', df['c_NSP_RT_kW'], bid_colors['NSP_RT']),
            ('-$c^{RD,DA}$', -df['c_RD_DA_kW'], bid_colors['RD_DA']), ('-$c^{RD,RT}$', -df['c_RD_RT_kW'], bid_colors['RD_RT']),
        ]
        _grouped_bars(ax, bid_components)
        ax.plot(x, df['p_up_bound_kW'], color='#F05A28', lw=1.1, label='$p^{up}$')
        ax.plot(x, -df['p_down_bound_kW'], color='#2C7FB8', lw=1.1, label='$-p^{down}$')
        ax.axhline(0, color='#555555', lw=0.55)
        ax = axes[2, col]
        obligation_components = [
            ('$p^{DA}$', df['p_DA_kW'], bid_colors['p_DA']), ('$p^{RT}$', df['p_RT_deviation_kW'], bid_colors['p_RT']),
            ('RU deployment', alpha['RU'] * df['c_RU_actual_kW'], bid_colors['RU_DA']),
            ('SP deployment', alpha['SP'] * df['c_SP_actual_kW'], bid_colors['SP_DA']),
            ('NSP deployment', alpha['NSP'] * df['c_NSP_actual_kW'], bid_colors['NSP_DA']),
            ('-RD deployment', -alpha['RD'] * df['c_RD_actual_kW'], bid_colors['RD_DA']),
        ]
        _signed_stack(ax, obligation_components)
        obligation = df['p_DA_kW'] + df['p_RT_deviation_kW'] + alpha['RU']*df['c_RU_actual_kW'] + alpha['SP']*df['c_SP_actual_kW'] + alpha['NSP']*df['c_NSP_actual_kW'] - alpha['RD']*df['c_RD_actual_kW']
        ax.plot(x, obligation, color='#111111', lw=1.15, label='Net obligation')
        ax.plot(x, df['p_up_bound_kW'], color='#F05A28', lw=0.9, label='$p^{up}$')
        ax.plot(x, -df['p_down_bound_kW'], color='#2C7FB8', lw=0.9, label='$-p^{down}$')
        ax.axhline(0, color='#555555', lw=0.55)
        ax = axes[3, col]
        ax.plot(x, 100*df['SOC'], color='#6A3D9A', lw=1.2, label='BESS SOC'); ax.axhline(5, color='#777777', ls='--', lw=0.7); ax.axhline(95, color='#777777', ls='--', lw=0.7)
        ax.set_ylim(0, 100)
        ax = axes[4, col]
        ax.plot(x, 1000*df['LMP_DA_$/kWh'], color='#4C78A8', lw=1.0, label='DA LMP')
        ax.plot(x, 1000*df['LMP_RT_$/kWh'], color='#E45756', lw=0.9, label='RT LMP')
        ax.plot(x, 1000*df['TOU_$/kWh'], color='#009E73', lw=1.0, label='TOU')
        ax.axhline(0, color='#555555', lw=0.55); ax.set_xticks(np.arange(0, 96, 16)); ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 4)]); ax.set_xlabel('Time')
    row_labels = ['Physical dispatch (kW)', 'Separate bids (kW)', 'Executed obligation (kW)', 'BESS SOC (%)', 'Price (US$/MWh)']
    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label)
        for ax in axes[row, :]: ax.grid(color='#bdbdbd', alpha=0.24, lw=0.55)
    for row in range(5):
        handles, labels = axes[row, 0].get_legend_handles_labels()
        if handles:
            ncol = min(max(2, int(np.ceil(len(handles)/2))), 6)
            axes[row, 2].legend(handles, labels, ncol=ncol, frameon=False, fontsize=7.2, loc='upper right', bbox_to_anchor=(1.0, 1.02))
    fig.suptitle(f'Executed Rolling and oracle validation — {day:%B %d, %Y}', fontsize=13, fontweight='bold', y=0.985)
    daily_path_out = PAPER_V28_DAILY / f'daily_validation_{day:%Y-%m-%d}.png'
    fig.savefig(daily_path_out, dpi=300, bbox_inches='tight', facecolor='white')
    if day in V28_SELECTED_DAYS:
        stem = 'fig_05_daily_validation_2025-06-03'
        fig.savefig(LATEX_V28_FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor='white')
        v28_manifest_rows.append({'figure': f'{stem}.png', 'formal_tag': GATE_E_TAG, 'paper_copy': True, 'source_files': 'Gate E executed traces and oracle dispatch', 'source_sha256': _sha256_v28(oracle_dispatch_path)})
    plt.close(fig)

for day in pd.date_range('2025-06-01', '2025-06-30', freq='D'):
    plot_v28_daily_validation(day)

v28_monthly.to_csv(PAPER_V28_ROOT / 'formal_monthly_financial.csv', index=False, float_format='%.6f')
v28_daily.to_csv(PAPER_V28_ROOT / 'formal_daily_financial.csv', index=False, float_format='%.6f')
pd.DataFrame(v28_manifest_rows).to_csv(PAPER_V28_ROOT / 'paper_figure_manifest.csv', index=False)
print('SEGAN v2.8 figures:', PAPER_V28_ROOT.relative_to(PROJECT))
print('Daily validation PNG files:', len(list(PAPER_V28_DAILY.glob('*.png'))))
print(v28_monthly[['Method', 'Total Revenue']].to_string(index=False))


SEGAN v2.8 figures: Paper_Figures/financial_comparison/2025-06/paper_v2.8
Daily validation PNG files: 30
                            Method  Total Revenue
          Rolling 24 h Persistence   10485.280000
              Rolling 24 h Perfect   10733.150000
Full billing-period Perfect oracle   12565.154573


In [ ]:
# SEGAN v3.0 paper exports (300 dpi); no temporary or executed-copy notebook.
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, FancyBboxPatch, FancyArrowPatch
from PIL import Image
P=Path.cwd(); R=P/'Results_Rolling'; ORACLE=P/'Validation_Results/June_2025/formal_runs/gate_f_20260813_price_resolution/rolling_perfect/oracle'; ORACLE_CERT=P/'Validation_Results/June_2025/formal_runs/gate_f_20260813_price_resolution_certified/rolling_perfect/oracle'; O=P/'Latex/paper3.0'; A=P/'Paper_Figures/financial_comparison/paper_v3.0'
O.mkdir(parents=True,exist_ok=True); A.mkdir(parents=True,exist_ok=True)
PAPER_MAX_WIDTH_PX=2400
def _write_paper_png(src,dst,max_width=PAPER_MAX_WIDTH_PX):
    with Image.open(src) as opened:
        mode='RGBA' if 'A' in opened.getbands() else 'RGB'
        im=opened.convert(mode)
    if im.width>max_width:
        new_h=int(round(im.height*max_width/im.width))
        im=im.resize((max_width,new_h),Image.Resampling.LANCZOS)
    im.save(dst,format='PNG',dpi=(300,300),optimize=True)
def _save_v3_figure(fig,dst):
    fig.savefig(dst,dpi=300,bbox_inches='tight',facecolor='white')
    _write_paper_png(dst,dst)
figure_sources=[
    (P/'Paper_Figures/static_sources/legacy_market_bidding_mpc_timeline.png','fig_06_legacy_market_bidding_mpc_timeline.png'),
    (R/'Plots/Solver_RT_Choices/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival_full/20250603/RT_6panel_2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival_full.png','fig_08_rolling_persistence_daily_6panel_2025-06-03.png'),
    (R/'Plots/Solver_RT_Choices/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival_full/20250603/RT_6panel_2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival_full.png','fig_09_rolling_perfect_daily_6panel_2025-06-03.png'),
]
for src,name in figure_sources:
    if not src.exists(): raise FileNotFoundError(src)
    _write_paper_png(src,O/name)
C={'navy':'#1F4E79','blue':'#4C78A8','teal':'#2A9D8F','orange':'#F4A261','gold':'#E9C46A','red':'#D95F59','purple':'#7A5195','gray':'#8A8F98','light':'#D9DDE3','ink':'#20242A','grid':'#D6D9DE'}
plt.rcParams.update({'font.size':9.5,'axes.titlesize':11,'axes.labelsize':9.5,'legend.fontsize':8.2,'axes.edgecolor':'#6E737B','axes.linewidth':0.75})
# Reproducible campus BTM architecture; passive PV/building have no wholesale branch.
fig,ax=plt.subplots(figsize=(12,5.8)); ax.set_xlim(0,12); ax.set_ylim(0,5.8); ax.axis('off')
def _arch_box(x,y,w,h,title,subtitle='',fc='white',ec='#667085',lw=1.3,rounding=.13,title_color='#20242A',fs=14.2,sub_fs=10.2):
    patch=FancyBboxPatch((x,y),w,h,boxstyle=f'round,pad=0.035,rounding_size={rounding}',facecolor=fc,edgecolor=ec,linewidth=lw)
    ax.add_patch(patch); ax.text(x+w/2,y+h*(.60 if subtitle else .5),title,ha='center',va='center',fontsize=fs,fontweight='bold',color=title_color)
    if subtitle: ax.text(x+w/2,y+h*.25,subtitle,ha='center',va='center',fontsize=sub_fs,color='#667085')
def _arch_arrow(a,b,color='#667085',lw=1.4,style='-|>',ls='-'):
    ax.add_patch(FancyArrowPatch(a,b,arrowstyle=style,mutation_scale=12,linewidth=lw,color=color,linestyle=ls,connectionstyle='arc3',shrinkA=2,shrinkB=2))
camp=FancyBboxPatch((.25,.30),6.70,5.20,boxstyle='round,pad=0.04,rounding_size=.18',facecolor='#FAFBFC',edgecolor=C['navy'],linewidth=1.7); ax.add_patch(camp)
ax.text(.55,5.25,'CAMPUS BEHIND-THE-METER PORTFOLIO',fontsize=15.5,fontweight='bold',color=C['navy'],va='center')
ax.text(.55,4.98,'Shared point of common coupling',fontsize=10.5,color='#667085',va='center')
_arch_box(2.82,4.04,1.28,.70,'PCC','SIGNED METER',ec=C['navy'],lw=1.7,rounding=.10,fs=14,sub_fs=8.8)
ax.plot([3.46,3.46],[1.58,4.02],color=C['ink'],lw=2.0); _arch_arrow((3.46,3.98),(3.46,4.04),color=C['ink'],lw=1.8)
ax.text(3.46,3.83,r'$p^{GI}=p^{load}+p^{EV}+p^{ch,BESS}-p^{dch,BESS}-p^{PV}$',ha='center',va='top',fontsize=10.5,color=C['ink'])
_arch_box(.62,2.65,2.05,.88,'Building demand','PASSIVE',fc='#EEF0F3',ec=C['gray'])
_arch_box(.62,1.45,2.05,.88,'PV generation','PASSIVE',fc='#E3F3EF',ec=C['teal'])
_arch_box(4.25,2.65,2.05,.88,'EV aggregation','CONTROLLABLE',fc='#FFF0E2',ec=C['orange'])
_arch_box(4.25,1.45,2.05,.88,'BESS','CONTROLLABLE',fc='#E8F0F7',ec=C['blue'])
_arch_arrow((2.67,3.09),(3.43,3.09),color=C['gray'],lw=1.45); _arch_arrow((2.67,1.89),(3.43,1.89),color=C['teal'],lw=1.45)
_arch_arrow((4.25,3.09),(3.49,3.09),color=C['orange'],lw=1.45); _arch_arrow((4.25,1.89),(3.49,1.89),color=C['blue'],lw=1.45,style='<|-|>')
_arch_box(2.05,.46,2.82,.70,'24-h rolling MPC','FORECAST - DISPATCH - SETTLEMENT',fc='#EAF0F6',ec=C['navy'],lw=1.7,rounding=.11,title_color=C['navy'],fs=14,sub_fs=8.8)
ax.plot([4.12,5.65,5.65],[1.16,1.28,2.63],color=C['orange'],lw=1.1,ls='--'); _arch_arrow((5.65,2.60),(5.65,2.65),color=C['orange'],lw=1.1,ls='--')
ax.plot([4.48,4.48],[1.16,1.42],color=C['blue'],lw=1.1,ls='--'); _arch_arrow((4.48,1.40),(4.48,1.45),color=C['blue'],lw=1.1,ls='--')
_arch_box(7.65,3.60,4.00,1.28,'Retail tariff','TOU ENERGY - PD - NCD',fc='#FFF7DE',ec=C['gold'],lw=1.7,rounding=.17,fs=15.5,sub_fs=9.8)
_arch_box(7.65,.42,4.00,1.82,'CAISO wholesale market','DA/RT ENERGY\nRU - RD - SP - NSP',fc='#F0EAF5',ec=C['purple'],lw=1.7,rounding=.17,fs=15,sub_fs=9.8)
_arch_arrow((4.10,4.41),(7.62,4.41),color=C['navy'],lw=1.8); _arch_arrow((7.62,3.96),(4.10,4.17),color=C['gold'],lw=1.35)
ax.text(5.86,4.55,'signed meter energy',fontsize=10.5,color=C['navy'],ha='center'); ax.text(5.82,3.98,'tariff signal',fontsize=10,color='#9A7620',ha='center')
_arch_arrow((4.87,.70),(7.62,.70),color=C['purple'],lw=1.8); _arch_arrow((7.62,1.16),(4.87,1.08),color=C['navy'],lw=1.45)
ax.text(6.24,.48,'DA bids - RT adjustments',fontsize=10.3,color=C['purple'],ha='center'); ax.text(6.24,1.30,'prices - awards - activation',fontsize=10,color=C['navy'],ha='center')
_save_v3_figure(fig,O/'fig_07_legacy_market_interface_context.png'); plt.close(fig)
cost=R/'Plots/Cost'; imp=pd.read_csv(R/'Dispatch/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival_full_implementation.csv'); imp['Interval start']=pd.to_datetime(imp['Interval start']); h=imp['Interval start'].dt.hour+imp['Interval start'].dt.minute/60
rate=np.full(len(imp),0.21421); rate[(h>=16)&(h<21)]=0.825; rate[h<6]=0.12071; e=imp['V1G_real [kWh]'].to_numpy(float); p=e/0.25; peak=(h>=16)&(h<21)
obs={'Method':'Observed V1G real','WM Revenue':0.0,'TOU Cost':-float(np.sum(rate*e)),'PD Cost':-3.05*float(np.max(p[peak])),'NCD Cost':-15.38*float(np.max(p)),'EV Revenue':0.4*float(np.sum(e))}; obs['Total Revenue']=sum(obs[k] for k in ['WM Revenue','TOU Cost','PD Cost','NCD Cost','EV Revenue']); rows=[obs]
for label,folder in [('24-h Persistence','2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival'),('24-h Perfect','2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival')]:
    summary=pd.read_csv(cost/folder/'monthly_financial_summary.csv'); match=summary[summary['Case / US$'].eq('Both')];
    if len(match)!=1: raise ValueError(f'{label}: expected exactly one full-market monthly row, found {len(match)}')
    r=match.iloc[0]; rows.append({'Method':label,**{k:float(r[k]) for k in ['Total Revenue','WM Revenue','TOU Cost','PD Cost','NCD Cost','EV Revenue']}})
cert=pd.read_csv(ORACLE_CERT/'oracle_joint_certificate.csv').iloc[0]
if not bool(cert['accepted']) or float(cert['certified_relative_gap'])>0.01: raise ValueError('Full-period oracle joint certificate does not meet the 1% criterion')
if Path(cert['feasible_solution_dir']).resolve()!=ORACLE.resolve(): raise ValueError('Joint certificate points to an unexpected oracle incumbent')
shutil.copy2(ORACLE_CERT/'oracle_joint_certificate.csv',A/'full_period_oracle_joint_certificate.csv')
r=pd.read_csv(ORACLE/'oracle_monthly_summary.csv').iloc[0]; rows.append({'Method':'Full-period oracle',**{k:float(r[k]) for k in ['Total Revenue','WM Revenue','TOU Cost','PD Cost','NCD Cost','EV Revenue']}})
m=pd.DataFrame(rows); m.to_csv(A/'formal_monthly_reference_mpc_oracle.csv',index=False,float_format='%.6f'); x=np.arange(len(m)); fig,ax=plt.subplots(1,2,figsize=(12.4,4.8),gridspec_kw={'width_ratios':[0.86,1.45]}); colors=[C['gray'],C['orange'],C['teal'],C['light']]
b=ax[0].bar(x,m['Total Revenue'],color=colors,edgecolor=C['ink'],linewidth=.55); b[-1].set_hatch('///'); ax[0].axhline(0,color=C['ink'],lw=.8); xt=['Observed\nV1G real','24-h\nPersistence','24-h\nPerfect','Full-period\noracle']; ax[0].set_xticks(x,xt); ax[0].set_ylabel('Net portfolio value (US dollars)'); ax[0].set_title('(a) Executed monthly value',loc='left',fontweight='bold')
for q,v in zip(b,m['Total Revenue']): ax[0].text(q.get_x()+q.get_width()/2,v+(260 if v>=0 else -430),f'{v:,.0f}',ha='center',va='bottom' if v>=0 else 'top',fontsize=8.3,fontweight='bold')
pos=np.zeros(len(m)); neg=np.zeros(len(m))
for lab,col,color in [('EV revenue','EV Revenue',C['navy']),('WM revenue','WM Revenue',C['teal']),('TOU','TOU Cost',C['orange']),('PD','PD Cost',C['gold']),('NCD','NCD Cost',C['red'])]:
    v=m[col].to_numpy(float); bot=np.where(v>=0,pos,neg); ax[1].bar(x,v,bottom=bot,color=color,width=.72,label=lab,edgecolor='white',linewidth=.35); pos+=np.where(v>=0,v,0); neg+=np.where(v<0,v,0)
ax[1].scatter(x,m['Total Revenue'],color=C['ink'],marker='D',s=28,zorder=5,label='Net value'); ax[1].axvline(2.5,color=C['gray'],ls='--',lw=.8); ax[1].axhline(0,color=C['ink'],lw=.8); ax[1].set_xticks(x,xt); ax[1].set_ylabel('Financial contribution (US dollars)'); ax[1].set_title('(b) Retail-wholesale value stack',loc='left',fontweight='bold'); ax[1].legend(ncol=3,frameon=False,loc='upper center',bbox_to_anchor=(.5,1.19))
for z in ax: z.grid(axis='y',color=C['grid'],alpha=.75,lw=.65); z.set_axisbelow(True)
fig.suptitle('Observed charging reference and optimized 24-h MPC',fontsize=13,fontweight='bold',y=1.01); fig.tight_layout()
_save_v3_figure(fig,O/'fig_03_monthly_reference_and_mpc_value.png')
plt.close(fig)
# Current Results_Rolling resource/service attribution (June full-market only).
decomp_cols=['WM EV Energy','WM EV Capacity','WM BESS Energy','WM BESS Capacity','NWM EV Energy','NWM EV Capacity','NWM BESS Energy','NWM BESS Capacity']
decomp_rows=[]
for label,folder in [('24-h Persistence','2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival'),('24-h Perfect','2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival')]:
    d=pd.read_csv(cost/folder/'daily_financial_detail.csv'); d['Date']=pd.to_datetime(d['Date']); d=d[(d['Date'].dt.year==2025)&(d['Date'].dt.month==6)&(d['Case']=='Both')]
    if len(d)!=30 or d['Date'].nunique()!=30: raise ValueError(f'{label}: expected 30 unique June full-market rows, found {len(d)}')
    row={'Method':label,'Total Revenue':float(d['Total Revenue'].sum())}; row.update({c:float(d[c].sum()) for c in decomp_cols}); decomp_rows.append(row)
decomp=pd.DataFrame(decomp_rows); decomp['Decomposition Residual']=decomp['Total Revenue']-decomp[decomp_cols].sum(axis=1); decomp.to_csv(A/'rolling_monthly_resource_service_decomposition.csv',index=False,float_format='%.6f')
group_colors={'WM EV':C['teal'],'WM BESS':C['blue'],'NWM EV':C['orange'],'NWM BESS':C['purple']}
fig,ax=plt.subplots(figsize=(8.2,4.5),constrained_layout=True); x=np.arange(len(decomp)); pos=np.zeros(len(x)); neg=np.zeros(len(x))
for col in decomp_cols:
    v=decomp[col].to_numpy(float); bottom=np.where(v>=0,pos,neg); group=col.rsplit(' ',1)[0]; hatch='////' if col.endswith('Capacity') else None; ax.bar(x,v,bottom=bottom,width=.58,color=group_colors[group],hatch=hatch,edgecolor='white',linewidth=.45); pos+=np.where(v>=0,v,0); neg+=np.where(v<0,v,0)
ax.scatter(x,decomp['Total Revenue'],color=C['ink'],marker='D',s=30,zorder=4); ax.axhline(0,color=C['ink'],lw=.8); ax.set_xticks(x,['24-h\nPersistence','24-h\nPerfect']); ax.set_ylabel('June contribution (US dollars)'); ax.grid(axis='y',color=C['grid'],alpha=.75,lw=.65); ax.set_axisbelow(True); ax.set_title('Executed-value attribution by resource and service',fontweight='bold',pad=42)
handles=[Patch(facecolor=v,label=k) for k,v in group_colors.items()]+[Patch(facecolor='#888888',label='Energy'),Patch(facecolor='#888888',hatch='////',label='Capacity')]; ax.legend(handles=handles,ncol=3,frameon=False,loc='upper center',bbox_to_anchor=(.5,1.20))
_save_v3_figure(fig,O/'fig_04_monthly_rolling_resource_service_decomposition.png')
plt.close(fig)
sens_path=P/'Sensitivity_Results/Rolling_24h/callrate_unitfix_multidate_comparison/callrate_multidate_financial_comparison.csv'; raw=pd.read_csv(sens_path); order=['Fixed 0.0','Fixed 0.1','Fixed 0.3','Fixed 0.5','Fixed 0.7','Fixed 1.0','CAISO attenuation']; raw['Alpha']=pd.Categorical(raw['Alpha'],order,ordered=True); raw=raw.sort_values(['Date','Alpha']); s=raw.groupby('Alpha',observed=True,sort=False).sum(numeric_only=True).reindex(order).reset_index(); raw.to_csv(A/'activation_factor_daily_financial.csv',index=False,float_format='%.6f'); s.to_csv(A/'activation_factor_three_day_financial.csv',index=False,float_format='%.6f'); x=np.arange(len(s)); xt=['0.0','0.1','0.3','0.5','0.7','1.0','CAISO\nattenuation']; fig,ax=plt.subplots(2,2,figsize=(12.2,7.5),constrained_layout=True)
date_colors=[C['navy'],C['orange'],C['teal']]
for color,(date,d) in zip(date_colors,raw.groupby('Date',sort=True)): d=d.set_index('Alpha').reindex(order); ax[0,0].plot(x,d['WM Revenue'],color=color,marker='o',lw=1.6,label=str(date))
ax[0,0].set_ylabel('Wholesale revenue (US dollars/day)'); ax[0,0].set_title('(a) Daily wholesale-market revenue',loc='left',fontweight='bold'); ax[0,0].legend(ncol=3,frameon=False,loc='upper center',bbox_to_anchor=(.5,1.20))
for color,(date,d) in zip(date_colors,raw.groupby('Date',sort=True)): d=d.set_index('Alpha').reindex(order); ax[0,1].plot(x,d['Total Revenue'],color=color,marker='o',lw=1.6,label=str(date))
ax[0,1].set_ylabel('Total net value (US dollars/day)'); ax[0,1].set_title('(b) Daily total net value',loc='left',fontweight='bold')
base=np.zeros(len(s))
for lab,col,color in [('BESS energy','WM BESS Energy',C['navy']),('EV energy','WM EV Energy',C['blue']),('BESS capacity','WM BESS Capacity',C['purple']),('EV capacity','WM EV Capacity',C['orange'])]: ax[1,0].bar(x,s[col],bottom=base,color=color,width=.72,label=lab,edgecolor='white',linewidth=.35); base+=s[col].to_numpy(float)
ax[1,0].set_ylabel('Three-day contribution (US dollars)'); ax[1,0].set_title('(c) Three-day WM decomposition',loc='left',fontweight='bold'); ax[1,0].legend(ncol=2,frameon=False,loc='upper center',bbox_to_anchor=(.5,1.20))
ref=float(s.loc[s['Alpha'].eq('CAISO attenuation'),'WM Revenue'].iloc[0]); d=s['WM Revenue']-ref; b=ax[1,1].bar(x,d,color=[C['teal']]*6+[C['gray']],width=.68); ax[1,1].axhline(0,color=C['ink'],lw=.8); ax[1,1].set_ylabel('Difference from CAISO (US dollars/three days)'); ax[1,1].set_title('(d) WM revenue relative to CAISO attenuation',loc='left',fontweight='bold')
for q,v in zip(b,d): ax[1,1].text(q.get_x()+q.get_width()/2,v+(.45 if v>=0 else -.45),f'{v:+.1f}',ha='center',va='bottom' if v>=0 else 'top',fontsize=7.8)
for z in ax.flat: z.set_xticks(x,xt); z.set_xlabel('RU/RD activation model'); z.grid(axis='y',color=C['grid'],alpha=.75,lw=.65); z.set_axisbelow(True)
fig.suptitle('Rolling 24-h MPC activation-factor sensitivity with passive PV/building retail meter',fontsize=13,fontweight='bold')
_save_v3_figure(fig,O/'fig_07_regulation_activation_sensitivity.png')
plt.close(fig); print('SEGAN v3.0 figures saved:',O); print(m[['Method','Total Revenue']].to_string(index=False))

In [ ]:
# SEGAN v3.0 appendix validation matrix with compact, row-specific legends.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
P=Path.cwd(); R=P/'Results_Rolling'; ORACLE=P/'Validation_Results/June_2025/formal_runs/gate_f_20260813_price_resolution/rolling_perfect/oracle'; O=P/'Latex/paper3.0'; A=P/'Paper_Figures/financial_comparison/paper_v3.0'
C={'navy':'#1F4E79','blue':'#4C78A8','teal':'#2A9D8F','orange':'#F4A261','red':'#D95F59','purple':'#7A5195','gray':'#8A8F98','ink':'#20242A','grid':'#D6D9DE'}
day=pd.Timestamp('2025-06-03'); roots={'24-h Persistence':R/'Validation_Traces/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival_full','24-h Perfect':R/'Validation_Traces/2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival_full'}
frames={}
for label,root in roots.items():
    f=pd.read_csv(root/f'rolling_validation_trace_{day:%Y%m%d}.csv'); f['Interval start']=pd.to_datetime(f['Interval start']); frames[label]=f.sort_values('Interval start').reset_index(drop=True)
f=pd.read_csv(ORACLE/'oracle_dispatch.csv'); f['Interval start']=pd.to_datetime(f['Interval start']); frames['Full-period oracle']=f[f['Interval start'].dt.normalize().eq(day)].sort_values('Interval start').reset_index(drop=True)
fig,ax=plt.subplots(5,3,figsize=(13.2,10.2),sharex='col',constrained_layout=False)
fig.subplots_adjust(left=0.065,right=0.985,bottom=0.065,top=0.93,wspace=0.17,hspace=0.24)
for j,label in enumerate(frames):
    f=frames[label]; x=np.arange(96); ax[0,j].set_title(label,fontweight='bold')
    ax[0,j].plot(x,f['p_GI_kW'],color=C['ink'],lw=1.2,label='Meter'); ax[0,j].plot(x,f['p_EV_kW'],color=C['orange'],lw=1,label='EV'); ax[0,j].plot(x,f['p_BESS_kW'],color=C['blue'],lw=1,label='BESS')
    w=.38; ax[1,j].bar(x-w/2,f['p_DA_kW'],width=w,color=C['blue'],label='DA energy'); ax[1,j].bar(x+w/2,f['p_RT_deviation_kW'],width=w,color=C['teal'],label='RT adjustment'); ax[1,j].plot(x,f['p_up_bound_kW'],color=C['red'],lw=1,label='Up limit'); ax[1,j].plot(x,-f['p_down_bound_kW'],color=C['navy'],lw=1,label='Down limit')
    q=f['p_DA_kW']+f['p_RT_deviation_kW']+.7*f['c_RU_actual_kW']-.7*f['c_RD_actual_kW']+.2*f['c_SP_actual_kW']+.2*f['c_NSP_actual_kW']
    ax[2,j].plot(x,q,color=C['ink'],lw=1.3,label='Net obligation'); ax[2,j].fill_between(x,0,q,color=C['purple'],alpha=.16); ax[2,j].plot(x,f['p_up_bound_kW'],color=C['red'],lw=.9,label='Up limit'); ax[2,j].plot(x,-f['p_down_bound_kW'],color=C['navy'],lw=.9,label='Down limit')
    ax[3,j].plot(x,100*f['SOC'],color=C['purple'],lw=1.3,label='BESS SOC'); ax[3,j].axhline(5,color=C['gray'],ls='--',lw=.7); ax[3,j].axhline(95,color=C['gray'],ls='--',lw=.7); ax[3,j].set_ylim(0,100)
    ax[4,j].plot(x,f['LMP_DA_$/kWh'],color=C['blue'],lw=1,label='DA LMP'); ax[4,j].plot(x,f['LMP_RT_$/kWh'],color=C['red'],lw=.9,label='RT LMP'); ax[4,j].plot(x,f['TOU_$/kWh'],color=C['teal'],lw=1,label='TOU')
    ax[4,j].set_xticks(np.arange(0,96,16),[f'{h:02d}:00' for h in range(0,24,4)]); ax[4,j].set_xlabel('Time')
labels=['Physical power (kW)','Energy bids and limits (kW)','Deployed obligation (kW)','BESS SOC (%)','Price (US dollars/kWh)']
for i,ylab in enumerate(labels):
    ax[i,0].set_ylabel(ylab)
    for z in ax[i,:]: z.axhline(0,color='#6E737B',lw=.5); z.grid(color=C['grid'],alpha=.7,lw=.55); z.set_axisbelow(True)
    h,l=ax[i,0].get_legend_handles_labels(); ax[i,2].legend(h,l,ncol=min(len(l),2),frameon=True,framealpha=.88,fontsize=7.4,loc='upper right')
fig.suptitle('Common-axis daily validation matrix - June 3, 2025',fontsize=13,fontweight='bold')
validation_png=O/'fig_08_daily_validation_matrix_2025-06-03.png'
fig.savefig(validation_png,dpi=300,bbox_inches='tight',facecolor='white')
with Image.open(validation_png) as opened:
    im=opened.convert('RGBA' if 'A' in opened.getbands() else 'RGB')
if im.width>2400:
    im=im.resize((2400,int(round(im.height*2400/im.width))),Image.Resampling.LANCZOS)
im.save(validation_png,format='PNG',dpi=(300,300),optimize=True)
plt.close(fig); print('v3.0 appendix validation matrix saved')

## Code-generated architecture, system specifications, and staged BTM comparison

This section is the source of record for the manuscript system diagram and the NTPLL/Center Hall power and financial comparison figures. `Building only` and `Building + PV` are deterministic retail-billing counterfactuals; only `Building + PV + BESS + EV` uses the existing validated Rolling 24-h MPC results. No optimization is rerun here.


In [ ]:
# Code-generated Figure 1, site power specifications, and staged retail/full-portfolio comparison.
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Patch
from PIL import Image

ROOT = Path.cwd().resolve()
if not (ROOT / '2025Data').exists():
    raise FileNotFoundError('Run this notebook from the UPSCALeDEV_2024 repository root')
# Canonical June site results now come directly from the main Rolling notebook.
# Sensitivity_Results remains an archive and is not a formal data dependency.
SITE_ROOT = ROOT / 'Results_Rolling/Site_Cases_2025'
SITE_INPUT = ROOT / '2025Data/Site_Data_2025'
SITE_RESULTS = SITE_ROOT / 'model_outputs'
SITE_TABLES = SITE_ROOT / 'tables'
SITE_FIGURES = SITE_ROOT / 'figures'
PAPER_REVIEW = ROOT / 'Paper_Figures/financial_comparison/paper_v3.2'
LATEX_FIGURES = ROOT / 'Latex/paper3.2/figures'
for directory in [SITE_TABLES, SITE_FIGURES, PAPER_REVIEW, LATEX_FIGURES]:
    directory.mkdir(parents=True, exist_ok=True)

DT_H = 0.25
SUMMER_PD_RATE = 3.05
NCD_RATE = 15.38
CHARGER_UNIT_LIMIT_KW = 1.664 / DT_H
BESS_POWER_KW = 250.0
BESS_ENERGY_KWH = 332.0
SITE_META = {
    'NTPLL': {
        'label': 'NTPLL', 'building_meters': 7, 'pv_meters': 3, 'evse': 34,
        'pv_capacity_proxy_kW': 55.33 + 54.88 + 56.72,
    },
    'Center_Hall': {
        'label': 'Center Hall', 'building_meters': 1, 'pv_meters': 1, 'evse': 38,
        'pv_capacity_proxy_kW': 206.15,
    },
}
COLORS = {
    'navy': '#1F4E79', 'blue': '#4C78A8', 'teal': '#2A9D8F',
    'orange': '#F4A261', 'gold': '#E9C46A', 'red': '#D95F59',
    'purple': '#7A5195', 'gray': '#8A8F98', 'light': '#EEF1F5', 'ink': '#20242A',
}

def save_site_figure(fig, filename, manuscript_name=None):
    local_path = SITE_FIGURES / filename
    review_path = PAPER_REVIEW / filename
    fig.savefig(local_path, dpi=300, bbox_inches='tight', facecolor='white')
    with Image.open(local_path) as raster:
        if raster.width > 2400:
            target_height = round(raster.height * 2400 / raster.width)
            resized = raster.resize((2400, target_height), Image.Resampling.LANCZOS)
            resized.save(local_path, dpi=(300, 300), optimize=True)
    shutil.copy2(local_path, review_path)
    if manuscript_name:
        shutil.copy2(local_path, LATEX_FIGURES / manuscript_name)
    return local_path

# -------------------------------------------------------------------------
# Figure 1: compact physical, control, and market architecture.
# Run only this block to regenerate the diagram; no optimization is required.
# -------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(15.1, 7.8))
ax.set_xlim(0,15.1); ax.set_ylim(0,7.8); ax.axis('off')
navy, teal, purple = '#17649A', '#159E83', '#8853B1'
building_color = '#946A43'
orange, blue, gold, ink = '#E47722', '#267FC1', '#AE7D0A', '#303B49'
def architecture_box(x,y,w,h,title,subtitle='',face='white',edge=navy,title_size=14):
    ax.add_patch(FancyBboxPatch((x,y),w,h,
        boxstyle='round,pad=0.035,rounding_size=0.12',
        facecolor=face,edgecolor=edge,linewidth=1.9,zorder=3))
    ax.text(x+w/2,y+h*(.72 if subtitle else .5),title,
        ha='center',va='center',fontsize=title_size,fontweight='bold',color=edge,zorder=4)
    if subtitle:
        ax.text(x+w/2,y+h*.29,subtitle,ha='center',va='center',
            fontsize=10.2,fontweight='semibold',color=ink,linespacing=1.25,zorder=4)
def architecture_arrow(points,color=ink,dashed=False,both=False,lw=1.5):
    if len(points)>2:
        ax.plot(*zip(*points[:-1]),color=color,lw=lw,
            ls='--' if dashed else '-',zorder=2)
    ax.add_patch(FancyArrowPatch(points[-2],points[-1],
        arrowstyle='<|-|>' if both else '-|>',mutation_scale=13,
        color=color,lw=lw,linestyle='--' if dashed else '-',zorder=2))
def architecture_label(x,y,text,color=ink,size=9.5):
    ax.text(x,y,text,ha='center',va='center',fontsize=size,
        fontweight='semibold',color=color,zorder=5,
        bbox=dict(facecolor='white',edgecolor='none',pad=1.8))
# One enclosing portfolio makes the common physical boundary immediately visible.
ax.add_patch(FancyBboxPatch((.20,.43),9.55,7.02,
    boxstyle='round,pad=0.04,rounding_size=.18',
    facecolor='#FAFBFC',edgecolor=navy,linewidth=2.2,zorder=0))
ax.text(.58,7.10,'CAMPUS BEHIND-THE-METER PORTFOLIO',
    fontsize=17,fontweight='bold',color=navy)
architecture_box(1.05,5.05,2.65,1.00,'Building','Passive gross demand',
    '#F2E2CF',building_color)
architecture_box(1.05,3.72,2.65,1.00,'PV','Passive generation',
    '#CEF1E5',teal)
architecture_box(1.05,2.29,2.65,1.16,'EV fleet',
    'Persistence / Perfect updater','#FFE0BD',orange)
architecture_box(1.05,.84,2.65,1.16,'BESS',
    'Bidirectional storage','#D8EBFF',blue)
# Physical arrows: demand consumes, PV produces, and storage is bidirectional.
ax.plot([.65,.65],[1.42,6.55],color=ink,lw=2.0,zorder=1)
ax.plot([.65,7.20],[6.55,6.55],color=ink,lw=2.0,zorder=1)
architecture_arrow([(.67,5.55),(1.01,5.55)],lw=2.0)
architecture_arrow([(1.01,4.22),(.67,4.22)],lw=2.0)
architecture_arrow([(.67,2.87),(1.01,2.87)],lw=2.0)
architecture_arrow([(.67,1.42),(1.01,1.42)],both=True,lw=2.0)
architecture_label(3.35,6.80,'Shared electrical bus',size=10)
architecture_box(5.10,5.15,4.25,1.03,'PCC and grid meter',
    'Net electricity import / export','white',navy,15)
architecture_arrow([(7.20,6.55),(7.20,6.22)],both=True,lw=2.0)

# Forecasts and device data are represented inside their resource blocks.
architecture_box(5.10,1.00,4.25,3.42,'Rolling 24-h MPC',
    '', '#D3EAF5',navy,16)
# Use explicit text anchors to keep the controller title above its content.
# Replace the automatic centered title with a header and a concise state summary.
ax.texts[-1].set_position((7.225,3.93))
ax.plot([5.43,9.02],[3.59,3.59],color='#B7CADB',lw=1.2,zorder=4)
ax.text(7.225,3.17,'DA and RT optimization\nEV and BESS dispatch',
    ha='center',va='center',fontsize=12,fontweight='semibold',color=navy,
    linespacing=1.55,zorder=4)
ax.text(7.225,1.85,'Carried between updates:\nSOC, remaining EV energy,\nbilling peaks and commitments',
    ha='center',va='center',fontsize=10.8,fontweight='semibold',color=ink,
    linespacing=1.4,zorder=4)
# Passive profiles are inputs only; there is no control arrow back to PV/building.
architecture_arrow([(3.74,5.50),(4.24,5.50),(4.24,4.65),(5.75,4.65),(5.75,4.46)],building_color,True)
architecture_arrow([(3.74,4.22),(5.06,4.22)],teal,True)
architecture_label(4.30,5.90,'Building\nprofile',building_color,9.6)
architecture_label(4.40,4.04,'PV profile',teal,9.6)
# Each controllable resource has distinct input and output arrows.
architecture_arrow([(3.74,3.12),(5.06,3.12)],orange,True)
architecture_label(4.40,3.40,'EV data / forecast',orange,9.0)
architecture_arrow([(5.06,2.61),(3.74,2.61)],orange)
architecture_label(4.40,2.41,'Dispatch',orange,9.6)
architecture_arrow([(3.74,1.78),(5.06,1.78)],blue,True)
architecture_label(4.40,1.98,'State',blue,9.3)
architecture_arrow([(5.06,1.22),(3.74,1.22)],blue)
architecture_label(4.40,1.03,'Dispatch',blue,9.6)
architecture_arrow([(8.25,5.11),(8.25,4.46)],navy,True)
architecture_label(8.25,4.80,'Meter feedback',navy,9.6)

architecture_box(11.72,5.02,3.05,1.48,'SDG&E retail tariff',
    'Time-of-use energy\nPeak / noncoincident demand','#FFF0B2',gold,14)
architecture_arrow([(9.39,5.70),(11.68,5.70)],navy,True)
architecture_label(10.70,6.02,'Metered net\ndemand',navy,9.7)
architecture_arrow([(11.70,5.15),(10.05,5.15),(10.05,4.12),(9.39,4.12)],gold,True)
architecture_label(10.80,4.72,'Tariff inputs',gold,10)
architecture_box(11.72,1.02,3.05,2.63,'CAISO wholesale\nmarket',
    'DA / RT energy and AS\nRU, RD, SP, NSP\n\nEV and BESS only','#EDDAFA',purple,16)
architecture_arrow([(9.39,2.81),(11.68,2.81)],navy)
architecture_label(10.70,3.20,'DA bids /\nRT adjustments',navy,9.7)
architecture_arrow([(11.68,1.56),(9.39,1.56)],purple,True)
architecture_label(10.70,1.07,'Prices, factors\nand awards',purple,9.5)
ax.text(7.55,.08,
    'Dashed arrows: information / settlement data     |     Colored solid arrows: decisions     |     Dark solid arrows: physical power',
    ha='center',va='center',fontsize=9.2,fontweight='semibold',color='#526070')
fig.tight_layout(pad=.5)
# Preserve established filenames and synchronize the manuscript copies.
_fig01_name = 'fig_01_system_architecture_v31.png'
_fig01_targets = [
    ROOT / 'Results_Rolling/Site_Cases_2025/figures' / _fig01_name,
    ROOT / 'Paper_Figures/financial_comparison/paper_v3.2' / _fig01_name,
    ROOT / 'Latex/paper3.2/figures' / _fig01_name,
    ROOT / 'Latex/paper3.3/figures' / _fig01_name,
]
for _target in _fig01_targets:
    _target.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(_fig01_targets[0], dpi=300, bbox_inches='tight', facecolor='white')
for _target in _fig01_targets[1:]:
    shutil.copy2(_fig01_targets[0], _target)
plt.close(fig)

# -------------------------------------------------------------------------
# System specifications from June inputs. PV capacity is an observed-power
# proxy from the reviewed 2024 site study, not an unverified nameplate value.
# -------------------------------------------------------------------------
spec_rows = []
site_btm = {}
for site, meta in SITE_META.items():
    btm = pd.read_csv(SITE_INPUT / f'{site}_BTM_2025_June_July_15min_QC.csv')
    btm['Interval start'] = pd.to_datetime(btm['Interval start'])
    june = btm[btm['Interval start'].between('2025-06-01', '2025-06-30 23:45:00')].copy()
    if len(june) != 30 * 96 or june['Interval start'].nunique() != 30 * 96:
        raise ValueError(f'{site}: expected 2880 unique June BTM intervals')
    ev = pd.read_csv(SITE_INPUT / f'{site}_EV_2025_QC.csv', low_memory=False)
    ev['Interval start'] = pd.to_datetime(ev['Interval start'])
    june_ev = ev[ev['Interval start'].between('2025-06-01', '2025-06-30 23:45:00')].copy()
    if set(ev['Type'].dropna().unique()) != {'LiteON SC48'}:
        raise ValueError(f'{site}: unexpected charger type in site input')
    observed_evse = int(ev['Station Name'].nunique())
    if observed_evse != meta['evse']:
        raise ValueError(f'{site}: configured {meta["evse"]} EVSE but found {observed_evse}')
    coincident_ev = pd.to_numeric(june_ev['Interval average demand kW'], errors='raise').groupby(june_ev['Interval start']).sum()
    station_peak_sum = pd.to_numeric(ev['Interval max demand kW'], errors='raise').groupby(ev['Station Name']).max().sum()
    spec_rows.append({
        'Site': meta['label'],
        'Building meters': meta['building_meters'],
        'PV meters': meta['pv_meters'],
        'EVSE': meta['evse'],
        'Charger model': 'LiteON SC48',
        'Charger unit model limit (kW)': CHARGER_UNIT_LIMIT_KW,
        'Aggregate charger model limit (kW)': meta['evse'] * CHARGER_UNIT_LIMIT_KW,
        'Observed station-peak sum (kW)': float(station_peak_sum),
        'June coincident EV peak (kW)': float(coincident_ev.max()),
        'PV observed-capacity proxy (kW)': meta['pv_capacity_proxy_kW'],
        'June observed PV peak (kW)': float(june['p_PV_kW'].max()),
        'June gross building peak (kW)': float(june['p_load_kW'].max()),
        'June native net-load peak (kW)': float(june['p_native_net_kW'].max()),
        'BESS power (kW)': BESS_POWER_KW,
        'BESS energy (kWh)': BESS_ENERGY_KWH,
    })
    site_btm[site] = june
specs = pd.DataFrame(spec_rows)
specs.to_csv(SITE_TABLES / 'site_system_specifications.csv', index=False, float_format='%.6f')
specs.to_csv(PAPER_REVIEW / 'site_system_specifications.csv', index=False, float_format='%.6f')

spec_metrics = [
    ('June gross building peak (kW)', 'Gross building peak', COLORS['gray']),
    ('PV observed-capacity proxy (kW)', 'PV capacity proxy', COLORS['gold']),
    ('June observed PV peak (kW)', 'June PV peak', COLORS['teal']),
    ('Aggregate charger model limit (kW)', 'EVSE model limit', COLORS['orange']),
    ('June coincident EV peak (kW)', 'June EV coincident peak', COLORS['red']),
    ('BESS power (kW)', 'BESS power', COLORS['blue']),
]
fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.1), constrained_layout=True)
for ax, (_, row) in zip(axes, specs.iterrows()):
    values = [float(row[column]) for column, _, _ in spec_metrics]
    labels = [label for _, label, _ in spec_metrics]
    colors = [color for _, _, color in spec_metrics]
    y = np.arange(len(labels))
    bars = ax.barh(y, values, color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(y, labels); ax.invert_yaxis(); ax.set_xlabel('Power (kW)')
    ax.set_title(f'{row["Site"]}: {int(row["Building meters"])} building meters, {int(row["PV meters"])} PV meters, {int(row["EVSE"])} EVSE', loc='left', fontweight='bold')
    ax.grid(axis='x', color='#D6D9DE', alpha=0.75, linewidth=0.65); ax.set_axisbelow(True)
    for bar, value in zip(bars, values):
        ax.text(value + max(values) * 0.015, bar.get_y() + bar.get_height() / 2, f'{value:,.1f}', va='center', fontsize=8.5)
    ax.set_xlim(0, max(values) * 1.19)
fig.suptitle('Site System Power Specifications and Observed June Peaks', fontsize=13.5, fontweight='bold')
save_site_figure(fig, 'site_system_power_specifications.png', 'fig_16_site_system_power_specifications.png')
plt.close(fig)

# -------------------------------------------------------------------------
# Explicit aggregate-meter audit. The raw 2025 PV file contains only five
# PV channels, and all five are embedded behind the ten building net meters.
# Therefore the legacy aggregate expression sum(net meter) - total PV double
# subtracts PV; the corrected aggregate PCC passive term is sum(net meter).
# -------------------------------------------------------------------------
raw_load = pd.read_csv(ROOT / '2025Data/Bldg_data/Bldg_load.csv', low_memory=False)
raw_pv = pd.read_csv(ROOT / '2025Data/PV_data/CSV_2025-08-06-18-42-25.csv', low_memory=False)
raw_load['Interval start'] = pd.to_datetime(raw_load['Timestamp']) - pd.Timedelta(minutes=15)
raw_pv['Interval start'] = pd.to_datetime(raw_pv['Timestamp']) - pd.Timedelta(minutes=15)
load_cols = [c for c in raw_load if c not in ['Timestamp', 'Interval start']]
pv_cols = [c for c in raw_pv if c not in ['Timestamp', 'Interval start']]
embedded_pv = {
    'HOUSING_BLDG_2.PV_SYSTEM#Real Power Mean#kW', 'HOUSING_BLDG_3.3HPV#Real Power Mean#kW',
    'HOUSING_BLDG_4.PV#Real Power Mean#kW', 'PV.Osler_Parking_PV_P8900#Real Power Mean#kW',
    'WARREN.FAH_P2320#Real Power Mean#kW',
}
if set(pv_cols) != embedded_pv:
    raise ValueError('Aggregate PV mapping changed; classify every channel before using the aggregate case')
june_pv = raw_pv[raw_pv['Interval start'].between('2025-06-01', '2025-06-30 23:45:00')][pv_cols].apply(pd.to_numeric, errors='coerce').clip(lower=0).sum(axis=1)
aggregate_audit = pd.DataFrame([{
    'building_meter_count': len(load_cols), 'PV_meter_count': len(pv_cols),
    'all_PV_channels_embedded_in_building_net_meters': True,
    'legacy_passive_PCC_formula': 'sum(building net meters) - total PV',
    'corrected_passive_PCC_formula': 'sum(building net meters)',
    'June_mean_double_subtraction_kW': float(june_pv.mean()),
    'June_peak_double_subtraction_kW': float(june_pv.max()),
    'legacy_aggregate_result_status': 'invalid for quantitative comparison until corrected aggregate rerun',
}])
aggregate_audit.to_csv(SITE_TABLES / 'aggregate_net_meter_PV_accounting_audit.csv', index=False, float_format='%.6f')

# -------------------------------------------------------------------------
# Building only -> Building + PV -> full optimized portfolio. The first two
# cases use the exact executed June TOU signal and monthly demand-charge rules.
# -------------------------------------------------------------------------
validated_full = pd.read_csv(SITE_TABLES / 'june_Both_FULL_BTM_Both_financial_comparison.csv')
financial_rows = []
def retail_counterfactual(power_kW, tou_rate, pd_mask):
    power_kW = np.asarray(power_kW, dtype=float)
    tou_rate = np.asarray(tou_rate, dtype=float)
    pd_mask = np.asarray(pd_mask, dtype=bool)
    row = {
        'WM Revenue': 0.0,
        'TOU Cost': -float(np.sum(tou_rate * power_kW * DT_H)),
        'PD Cost': -float(SUMMER_PD_RATE * np.max(power_kW[pd_mask])),
        'NCD Cost': -float(NCD_RATE * np.max(power_kW)),
        'EV Revenue': 0.0,
    }
    row['Total Revenue'] = sum(row.values())
    return row
for site, meta in SITE_META.items():
    trace_dir = SITE_RESULTS / site / 'FULL_BTM/Validation_Traces/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival_full'
    trace_files = sorted(trace_dir.glob('rolling_validation_trace_202506*.csv'))
    if len(trace_files) != 30:
        raise ValueError(f'{site}: expected 30 June validation traces, found {len(trace_files)}')
    tariff = pd.concat([pd.read_csv(path, usecols=['Interval start', 'TOU_$/kWh']) for path in trace_files], ignore_index=True)
    tariff['Interval start'] = pd.to_datetime(tariff['Interval start'])
    tariff = tariff.sort_values('Interval start').drop_duplicates('Interval start')
    btm = site_btm[site].sort_values('Interval start')
    aligned = btm.merge(tariff, on='Interval start', how='inner', validate='one_to_one')
    if len(aligned) != 30 * 96:
        raise ValueError(f'{site}: BTM/tariff alignment is incomplete')
    on_peak_rate = float(aligned['TOU_$/kWh'].max())
    pd_mask = np.isclose(aligned['TOU_$/kWh'], on_peak_rate)
    for scenario, power_column in [('Building only', 'p_load_kW'), ('Building + PV', 'p_native_net_kW')]:
        financial_rows.append({
            'site': site, 'Scenario': scenario, 'Forecast': 'Not applicable',
            **retail_counterfactual(aligned[power_column], aligned['TOU_$/kWh'], pd_mask),
        })
    full_rows = validated_full[validated_full['site'].eq(site)]
    for _, source in full_rows.iterrows():
        financial_rows.append({
            'site': site, 'Scenario': 'Building + PV + BESS + EV', 'Forecast': source['forecast'],
            **{column: float(source[column]) for column in ['Total Revenue', 'WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost', 'EV Revenue']},
        })
staged = pd.DataFrame(financial_rows)
staged['Financial identity rounding residual'] = staged['Total Revenue'] - staged[['WM Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost', 'EV Revenue']].sum(axis=1)
# The validated site monthly table stores each component to cents; summing 30
# daily rounded rows can leave a few cents of monthly display-only residual.
if staged['Financial identity rounding residual'].abs().max() > 0.30:
    raise ValueError('Staged financial comparison exceeds the 30-day rounding allowance')
staged.to_csv(SITE_TABLES / 'site_resource_addition_financial_comparison.csv', index=False, float_format='%.6f')
staged.to_csv(PAPER_REVIEW / 'site_resource_addition_financial_comparison.csv', index=False, float_format='%.6f')

plot_order = [
    ('Building only', 'Not applicable', 'Building only'),
    ('Building + PV', 'Not applicable', 'Building + PV'),
    ('Building + PV + BESS + EV', 'Persistence', '24-h MPC\nPersistence forecast'),
    ('Building + PV + BESS + EV', 'Perfect', '24-h MPC\nPerfect forecast'),
]
component_order = ['WM Revenue', 'EV Revenue', 'TOU Cost', 'PD Cost', 'NCD Cost']
component_colors = {
    'WM Revenue': COLORS['teal'], 'EV Revenue': COLORS['navy'],
    'TOU Cost': COLORS['orange'], 'PD Cost': COLORS['gold'], 'NCD Cost': COLORS['red'],
}
fig, axes = plt.subplots(1, 2, figsize=(13.0, 6.0), constrained_layout=False)
fig.subplots_adjust(left=0.08, right=0.985, bottom=0.19, top=0.76, wspace=0.18)
for ax, site in zip(axes, SITE_META):
    site_rows = staged[staged['site'].eq(site)].set_index(['Scenario', 'Forecast'])
    rows = pd.DataFrame([site_rows.loc[(scenario, forecast)] for scenario, forecast, _ in plot_order])
    x = np.arange(len(rows)); positive = np.zeros(len(rows)); negative = np.zeros(len(rows))
    for component in component_order:
        values = rows[component].to_numpy(float)
        bottom = np.where(values >= 0, positive, negative)
        ax.bar(x, values, bottom=bottom, width=0.67, color=component_colors[component], edgecolor='white', linewidth=0.45, label=component)
        positive += np.where(values >= 0, values, 0.0); negative += np.where(values < 0, values, 0.0)
    net = rows['Total Revenue'].to_numpy(float)
    ax.scatter(x, net, color=COLORS['ink'], marker='D', s=28, zorder=5, label='Net financial outcome')
    ax.axhline(0, color=COLORS['ink'], linewidth=0.8)
    ax.set_xticks(x, [label for _, _, label in plot_order]); ax.set_ylabel('June financial outcome (US dollars)')
    ax.set_title(SITE_META[site]['label'], loc='left', fontweight='bold')
    ax.grid(axis='y', color='#D6D9DE', alpha=0.75, linewidth=0.65); ax.set_axisbelow(True)
    scale = max(np.abs(net).max(), 1.0)
    lower = min(float(negative.min()), float(net.min()))
    upper = max(float(positive.max()), 0.0)
    ax.set_ylim(lower - 0.09 * scale, upper + 0.10 * scale)
    for xi, value in zip(x, net):
        ax.annotate(f'{value:,.0f}', (xi, value), xytext=(0, 7 if value >= 0 else -9), textcoords='offset points', ha='center', va='bottom' if value >= 0 else 'top', fontsize=8.0, fontweight='bold')
handles = [Patch(facecolor=component_colors[c], label=c) for c in component_order]
handles.append(plt.Line2D([0], [0], marker='D', color='none', markerfacecolor=COLORS['ink'], label='Net financial outcome'))
fig.legend(handles=handles, ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 0.88))
fig.suptitle('June Financial Outcomes across Resource Configurations', fontsize=13.5, fontweight='bold', y=0.975)
save_site_figure(fig, 'site_resource_addition_financial_comparison.png', 'fig_17_site_resource_addition_financial_comparison.png')
plt.close(fig)

print('Generated architecture, power-specification, aggregate-audit, and staged-comparison artifacts.')
display(specs.round(3))
display(aggregate_audit.round(3))
display(staged.round(2))


### Site daily grid-import trajectories and campus maps

The daily comparison uses June 1 as the first evaluation day and June 6 as a later high-information day. June 6 was selected because it maximizes the two-site sum of the absolute 15-min difference between the executed Persistence and Perfect $p^{GI}$ trajectories over the remaining June dates. The campus-location figure is generated by code from OpenStreetMap tiles and fixed feature coordinates, with labels cross-checked against official UC San Diego campus maps.

In [ ]:
# Code-generated daily p_GI comparison and UC San Diego site-location maps.
import math
from io import BytesIO
import time
import requests
from PIL import Image
from matplotlib.lines import Line2D

DAILY_DATES = [pd.Timestamp('2025-06-01'), pd.Timestamp('2025-06-06')]
FORECAST_CASES = {
    'Persistence': '2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival_full',
    'Perfect': '2025_PerfectSessionkWh_PerfectNumbEV_PerfectatArrival_full',
}
selection_rows = []
for site in SITE_META:
    figure_data = {}
    btm = pd.read_csv(SITE_INPUT / f'{site}_BTM_2025_June_July_15min_QC.csv')
    btm['Interval start'] = pd.to_datetime(btm['Interval start'])
    for date in DAILY_DATES:
        day_btm = btm[btm['Interval start'].dt.normalize().eq(date)].copy()
        if len(day_btm) != 96:
            raise ValueError(f'{site} {date.date()}: expected 96 BTM intervals, found {len(day_btm)}')
        day_btm['hour'] = day_btm['Interval start'].dt.hour + day_btm['Interval start'].dt.minute / 60
        figure_data[date] = {'btm': day_btm}
        for forecast, run_name in FORECAST_CASES.items():
            trace_path = SITE_RESULTS / site / 'FULL_BTM/Validation_Traces' / run_name / f'rolling_validation_trace_{date:%Y%m%d}.csv'
            trace = pd.read_csv(trace_path, usecols=['Interval start', 'p_GI_kW'])
            trace['Interval start'] = pd.to_datetime(trace['Interval start'])
            trace['hour'] = trace['Interval start'].dt.hour + trace['Interval start'].dt.minute / 60
            if len(trace) != 96 or trace['Interval start'].nunique() != 96:
                raise ValueError(f'{site} {date.date()} {forecast}: incomplete executed trace')
            figure_data[date][forecast] = trace
    plotted_power = []
    for date in DAILY_DATES:
        plotted_power.extend(figure_data[date]['btm']['p_load_kW'].to_numpy(float))
        plotted_power.extend(figure_data[date]['btm']['p_native_net_kW'].to_numpy(float))
        plotted_power.extend(figure_data[date]['Persistence']['p_GI_kW'].to_numpy(float))
        plotted_power.extend(figure_data[date]['Perfect']['p_GI_kW'].to_numpy(float))
    power_min, power_max = float(np.min(plotted_power)), float(np.max(plotted_power))
    power_pad = max(0.08 * (power_max - power_min), 1.0)
    plot_lower, plot_upper = power_min - power_pad, power_max + power_pad

    fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.15), sharex=True, sharey=True)
    for ax, date in zip(axes, DAILY_DATES):
        day = figure_data[date]
        ax.plot(day['btm']['hour'], day['btm']['p_load_kW'], color=COLORS['gray'], linewidth=1.55, linestyle=':', label='Building only', zorder=2)
        ax.plot(day['btm']['hour'], day['btm']['p_native_net_kW'], color=COLORS['teal'], linewidth=1.75, linestyle='-.', label='Building + PV', zorder=3)
        for forecast, color, linestyle in [('Persistence', COLORS['blue'], '-'), ('Perfect', COLORS['orange'], '--')]:
            trace = day[forecast]
            ax.plot(trace['hour'], trace['p_GI_kW'], color=color, linewidth=2.05, linestyle=linestyle, label=f'24-h MPC: {forecast}', zorder=4)
        if plot_lower <= 0 <= plot_upper:
            ax.axhline(0, color=COLORS['ink'], linewidth=0.8, zorder=1)
        ax.set_ylim(plot_lower, plot_upper)
        ax.set_xlim(0, 24); ax.set_xticks([0, 4, 8, 12, 16, 20, 24])
        ax.set_xlabel('Hour of day')
        ax.set_title(f'{date:%B %-d, %Y}', loc='left', fontweight='bold')
        ax.grid(color='#D6D9DE', linewidth=0.6, alpha=0.75); ax.set_axisbelow(True)
    axes[0].set_ylabel('Signed PCC power, $p^{GI}$ (kW)', fontweight='semibold')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.91), fontsize=8.8)
    fig.suptitle(f'{SITE_META[site]["label"]}: Daily PCC Power across Resource Configurations', fontsize=13.5, fontweight='bold', y=0.995)
    fig.subplots_adjust(left=0.075, right=0.985, bottom=0.16, top=0.74, wspace=0.10)
    slug = site.lower()
    save_site_figure(fig, f'{slug}_staged_pgi_daily_timeseries.png', f'fig_{18 if site == "NTPLL" else 19:02d}_{slug}_staged_pgi_daily_timeseries.png')
    plt.close(fig)
    for date in DAILY_DATES:
        p = figure_data[date]['Persistence']['p_GI_kW'].to_numpy(float)
        q = figure_data[date]['Perfect']['p_GI_kW'].to_numpy(float)
        selection_rows.append({'site': site, 'date': date.date(), 'mean_absolute_Persistence_Perfect_difference_kW': np.mean(np.abs(p - q)), 'maximum_absolute_difference_kW': np.max(np.abs(p - q))})
selection = pd.DataFrame(selection_rows)
selection.to_csv(SITE_TABLES / 'daily_pgi_figure_date_summary.csv', index=False, float_format='%.6f')
selection.to_csv(PAPER_REVIEW / 'daily_pgi_figure_date_summary.csv', index=False, float_format='%.6f')

# Geographic feature coordinates are OpenStreetMap feature centroids. The
# model-to-location mapping is also checked against the official UCSD NTPLLN
# neighborhood map and the official UCSD main-campus map.
map_points = pd.DataFrame([
    {'site': 'NTPLL', 'category': 'Building load', 'label': 'RWAC (Bldg 1)', 'lat': 32.8804974, 'lon': -117.2410403, 'osm_feature': 'ways/893580777+893580778', 'model_mapping': 'E8801 BLDG 1 MAIN; location shown, but no channel is present in the current 2025 load input'},
    {'site': 'NTPLL', 'category': 'Building load', 'label': 'Catalyst (Bldg 2)', 'lat': 32.8807049, 'lon': -117.2418075, 'osm_feature': 'way/893580776', 'model_mapping': 'HOUSING_BLDG_2'},
    {'site': 'NTPLL', 'category': 'Building load', 'label': 'Kaleidoscope (Bldg 3)', 'lat': 32.8806291, 'lon': -117.2430327, 'osm_feature': 'way/893580774', 'model_mapping': 'HOUSING_BLDG_3'},
    {'site': 'NTPLL', 'category': 'Building load', 'label': 'Tapestry (Bldg 4)', 'lat': 32.8799540, 'lon': -117.2430416, 'osm_feature': 'way/893580775', 'model_mapping': 'HOUSING_BLDG_4'},
    {'site': 'NTPLL', 'category': 'Building load', 'label': 'Mosaic (Bldg 5)', 'lat': 32.8799783, 'lon': -117.2419074, 'osm_feature': 'way/911401513', 'model_mapping': 'HOUSING_BLDG_5'},
    {'site': 'NTPLL', 'category': 'Building load', 'label': 'The Jeannie (Bldg 6)', 'lat': 32.8799047, 'lon': -117.2412128, 'osm_feature': 'way/911401512', 'model_mapping': 'HOUSING_BLDG_6'},
    {'site': 'NTPLL', 'category': 'Building load', 'label': 'Scholars structure (Bldg 7)', 'lat': 32.8803346, 'lon': -117.2416709, 'osm_feature': 'way/989774306', 'model_mapping': 'HOUSING_BLDG_7.MSD7 + SS1 (two meter channels)'},
    {'site': 'NTPLL', 'category': 'PV', 'label': 'PV Bldg 2', 'lat': 32.8807049, 'lon': -117.2418075, 'osm_feature': 'way/893580776', 'model_mapping': 'HOUSING_BLDG_2.PV_SYSTEM'},
    {'site': 'NTPLL', 'category': 'PV', 'label': 'PV Bldg 3', 'lat': 32.8806291, 'lon': -117.2430327, 'osm_feature': 'way/893580774', 'model_mapping': 'HOUSING_BLDG_3.3HPV'},
    {'site': 'NTPLL', 'category': 'PV', 'label': 'PV Bldg 4', 'lat': 32.8799540, 'lon': -117.2430416, 'osm_feature': 'way/893580775', 'model_mapping': 'HOUSING_BLDG_4.PV'},
    {'site': 'NTPLL', 'category': 'EVSE', 'label': 'Scholars Parking (34 EVSE)', 'lat': 32.8803346, 'lon': -117.2416709, 'osm_feature': 'way/989774306', 'model_mapping': 'UCSD Scholars Parking Structure'},
    {'site': 'Center_Hall', 'category': 'Building load', 'label': 'Center Hall', 'lat': 32.8777123, 'lon': -117.2370703, 'osm_feature': 'way/31816858', 'model_mapping': 'University_Center.Center_Hall_E2451'},
    {'site': 'Center_Hall', 'category': 'PV', 'label': 'Osler/South Parking PV', 'lat': 32.8740136, 'lon': -117.2371457, 'osm_feature': 'way/587047714', 'model_mapping': 'PV.Osler_Parking_PV_P8900'},
    {'site': 'Center_Hall', 'category': 'EVSE', 'label': 'South Parking (38 EVSE)', 'lat': 32.8740136, 'lon': -117.2371457, 'osm_feature': 'way/587047714', 'model_mapping': 'UCSD South Parking Structure'},
])
map_points['included_in_2025_model'] = ~map_points['label'].eq('RWAC (Bldg 1)')
map_points['coordinate_source'] = 'OpenStreetMap feature centroid'
map_points['official_cross_check'] = np.where(map_points['site'].eq('NTPLL'), 'UCSD NTPLLN neighborhood map', 'UCSD main-campus map')
map_points.to_csv(SITE_TABLES / 'site_map_locations.csv', index=False)
map_points.to_csv(PAPER_REVIEW / 'site_map_locations.csv', index=False)

def tile_xy(lat, lon, zoom):
    n = 2 ** zoom
    x = (lon + 180.0) / 360.0 * n
    lat_rad = math.radians(lat)
    y = (1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n
    return x, y
def tile_lon(x, zoom):
    return x / (2 ** zoom) * 360.0 - 180.0
def tile_lat(y, zoom):
    return math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * y / (2 ** zoom)))))
def osm_basemap(bounds, zoom=18):
    lon_min, lon_max, lat_min, lat_max = bounds
    x0 = math.floor(tile_xy(lat_min, lon_min, zoom)[0]); x1 = math.floor(tile_xy(lat_max, lon_max, zoom)[0])
    y0 = math.floor(tile_xy(lat_max, lon_min, zoom)[1]); y1 = math.floor(tile_xy(lat_min, lon_max, zoom)[1])
    canvas = Image.new('RGB', ((x1 - x0 + 1) * 256, (y1 - y0 + 1) * 256), 'white')
    session = requests.Session(); session.headers.update({'User-Agent': 'UPSCALeDEV academic site-map figure/1.0'})
    for x in range(x0, x1 + 1):
        for y in range(y0, y1 + 1):
            response = session.get(f'https://tile.openstreetmap.org/{zoom}/{x}/{y}.png', timeout=30)
            response.raise_for_status(); time.sleep(0.08)
            with Image.open(BytesIO(response.content)) as tile:
                canvas.paste(tile.convert('RGB'), ((x - x0) * 256, (y - y0) * 256))
    extent = [tile_lon(x0, zoom), tile_lon(x1 + 1, zoom), tile_lat(y1 + 1, zoom), tile_lat(y0, zoom)]
    return np.asarray(canvas), extent

map_bounds = {
    'NTPLL': (-117.24345, -117.24075, 32.87962, 32.88098),
    'Center_Hall': (-117.24145, -117.23275, 32.87362, 32.87812),
}
marker_style = {
    'Building load': dict(marker='s', color=COLORS['navy'], size=54),
    'PV': dict(marker='^', color=COLORS['teal'], size=62),
    'EVSE': dict(marker='o', color=COLORS['orange'], size=72),
}
def add_scale_bar(ax, bounds, length_m):
    lon_min, lon_max, lat_min, lat_max = bounds
    mean_lat = 0.5 * (lat_min + lat_max)
    dx = length_m / (111320.0 * math.cos(math.radians(mean_lat)))
    x1 = lon_max - 0.055 * (lon_max - lon_min)
    x0 = x1 - dx
    y = lat_min + 0.065 * (lat_max - lat_min)
    cap = 0.018 * (lat_max - lat_min)
    ax.plot([x0, x1], [y, y], color='white', linewidth=4.0, solid_capstyle='butt', zorder=7)
    ax.plot([x0, x1], [y, y], color=COLORS['ink'], linewidth=1.8, solid_capstyle='butt', zorder=8)
    for x in [x0, x1]:
        ax.plot([x, x], [y - cap, y + cap], color='white', linewidth=3.4, zorder=7)
        ax.plot([x, x], [y - cap, y + cap], color=COLORS['ink'], linewidth=1.4, zorder=8)
    ax.text(0.5 * (x0 + x1), y + 1.55 * cap, f'{length_m:g} m', ha='center', va='bottom', fontsize=8.0, fontweight='bold', color=COLORS['ink'], zorder=9, bbox=dict(facecolor='white', edgecolor='none', alpha=0.78, pad=0.35))

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.45))
for ax, site in zip(axes, ['NTPLL', 'Center_Hall']):
    bounds = map_bounds[site]; image, extent = osm_basemap(bounds)
    ax.imshow(image, extent=extent, origin='upper', interpolation='bilinear')
    points = map_points[map_points['site'].eq(site)]
    for category, style in marker_style.items():
        sub = points[points['category'].eq(category) & points['included_in_2025_model']]
        ax.scatter(sub['lon'], sub['lat'], marker=style['marker'], s=style['size'], color=style['color'], edgecolor='white', linewidth=0.8, zorder=4)
    map_only = points[~points['included_in_2025_model']]
    ax.scatter(map_only['lon'], map_only['lat'], marker='s', s=58, facecolor='white', edgecolor=COLORS['navy'], linewidth=1.5, zorder=5)
    labels = points[points['category'].eq('Building load')].copy()
    if site == 'NTPLL':
        offsets = [(6, -17), (5, 7), (5, -13), (5, -13), (5, 7), (5, -13), (5, 7)]
        for (_, row), offset in zip(labels.iterrows(), offsets):
            ax.annotate(row['label'], (row['lon'], row['lat']), xytext=offset, textcoords='offset points', fontsize=7.0, fontweight='semibold', color=COLORS['ink'], zorder=5, bbox=dict(facecolor='white', edgecolor='none', alpha=0.74, pad=0.4))
        ax.annotate('34 EVSE', (-117.2416709, 32.8803346), xytext=(8, -25), textcoords='offset points', fontsize=8.0, fontweight='bold', color=COLORS['orange'], bbox=dict(facecolor='white', edgecolor='none', alpha=0.82, pad=0.5))
    else:
        ax.annotate('Center Hall', (-117.2370703, 32.8777123), xytext=(8, 8), textcoords='offset points', fontsize=8.2, fontweight='bold', color=COLORS['navy'], bbox=dict(facecolor='white', edgecolor='none', alpha=0.82, pad=0.5))
        ax.annotate('South/Osler Parking\nPV + 38 EVSE', (-117.2371457, 32.8740136), xytext=(8, 8), textcoords='offset points', fontsize=8.2, fontweight='bold', color=COLORS['ink'], bbox=dict(facecolor='white', edgecolor='none', alpha=0.82, pad=0.5))
    ax.set_xlim(bounds[0], bounds[1]); ax.set_ylim(bounds[2], bounds[3])
    ax.set_aspect(1.0 / math.cos(math.radians(0.5 * (bounds[2] + bounds[3]))))
    ax.set_title(SITE_META[site]['label'], fontweight='bold', pad=5); ax.set_xticks([]); ax.set_yticks([])
    add_scale_bar(ax, bounds, 100 if site == 'NTPLL' else 250)
    ax.text(0.01, 0.01, 'Basemap: © OpenStreetMap contributors', transform=ax.transAxes, fontsize=6.8, color='#4B5563', bbox=dict(facecolor='white', edgecolor='none', alpha=0.78, pad=1.0))
legend = [Line2D([0], [0], marker=v['marker'], linestyle='none', markerfacecolor=v['color'], markeredgecolor='white', markersize=8, label=k) for k, v in marker_style.items()]
legend.append(Line2D([0], [0], marker='s', linestyle='none', markerfacecolor='white', markeredgecolor=COLORS['navy'], markeredgewidth=1.4, markersize=7, label='Building location without 2025 meter input'))
fig.legend(handles=legend, loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.91), fontsize=8.8)
fig.suptitle('UC San Diego Site Locations and Modeled Resources', fontsize=13.5, fontweight='bold', y=0.985)
fig.subplots_adjust(left=0.015, right=0.995, bottom=0.025, top=0.78, wspace=0.025)
save_site_figure(fig, 'ucsd_site_location_maps.png', 'fig_20_ucsd_site_location_maps.png')
plt.close(fig)

print('Generated staged daily p_GI figures and the code-generated site-location map.')
display(selection.round(3))
display(map_points[['site', 'category', 'label', 'model_mapping']])


## Site-specific cross-month comparisons
March–July 2025 now uses the complete chronological March-onward run for both sites and forecasts. February baseline dispatch seeds March; every later month uses preceding executed dispatch. SOC starts/ends at 50% and demand-charge thresholds reset each billing month. Sources are recorded in the provenance CSV. All figures have titles and are generated at 300 dpi.


In [ ]:
# Site-specific cross-month publication comparison
# Reuses saved outputs; prefer newly validated chronological month runs.
from pathlib import Path
import calendar, shutil, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator

ROOT=Path.cwd().resolve()
STUDY=ROOT/'Results_Rolling/Site_Cases_2025/as_settlement_fix_20260912'
MONTHS=[3,4,5,6,7]
SITES=['NTPLL','Center_Hall']
FORECASTS=['Persistence','Perfect']
EXPORT=STUDY/'monthly_runs/cross_month_comparison'
REVIEW=ROOT/'Paper_Figures/financial_comparison/site_cross_month_2025'
LATEX=ROOT/'Latex/paper3.5/figures'
for p in [EXPORT,REVIEW,LATEX]:p.mkdir(parents=True,exist_ok=True)
def period_root(month):
    new=STUDY/'monthly_runs'/f'2025-{month:02d}'
    return new if (new/f'2025{month:02d}_month_Both_FULL_BTM_Both_manifest.json').exists() or month!=6 else STUDY
def is_chained(month):
    p=period_root(month)/f'2025{month:02d}_month_Both_FULL_BTM_Both_manifest.json'
    return p.exists() and json.loads(p.read_text()).get('chain_start_month')==3
def tag(month):return 'june_Both_FULL_BTM_Both' if period_root(month)==STUDY else f'2025{month:02d}_month_Both_FULL_BTM_Both'
def save_figure(fig,name):
    fig.savefig(EXPORT/name,dpi=300,bbox_inches='tight',pad_inches=.08,facecolor='white')
    for folder in [REVIEW,LATEX]:shutil.copy2(EXPORT/name,folder/name)
    plt.close(fig);print(EXPORT/name)

components=['WM Revenue','EV Revenue','TOU Cost','PD Cost','NCD Cost']
palette={'WM Revenue':'#2A9D8F','EV Revenue':'#1F4E79','TOU Cost':'#F4A261','PD Cost':'#E9C46A','NCD Cost':'#D95F59'}
forecast_colors={'Persistence':'#D48742','Perfect':'#276B92'}
rows=[];daily_rows=[];profiles={};provenance=[]
for month in MONTHS:
    period=period_root(month)
    checks=pd.read_csv(period/'tables'/f'{tag(month)}_validation_summary.csv')
    assert len(checks)==4 and checks['PASS'].astype(str).str.lower().eq('true').all(), 'Incomplete validated month'
    for site in SITES:
        site_base=period/'model_outputs'/site/'FULL_BTM'
        reference=None
        for forecast in FORECASTS:
            key=f'2025_{forecast}SessionkWh_{forecast}NumbEV_PerfectatArrival'
            path=site_base/'Dispatch'/f'{key}_full_implementation.csv'
            impl=pd.read_csv(path);impl['Interval start']=pd.to_datetime(impl['Interval start'])
            expected=pd.date_range(f'2025-{month:02d}-01',periods=calendar.monthrange(2025,month)[1]*96,freq='15min')
            assert pd.DatetimeIndex(impl['Interval start']).equals(expected),str(path)
            traces=pd.concat([pd.read_csv(site_base/'Validation_Traces'/f'{key}_full'/f'rolling_validation_trace_2025{month:02d}{day:02d}.csv') for day in range(1,calendar.monthrange(2025,month)[1]+1)],ignore_index=True)
            assert len(traces)==len(expected)
            assert abs(traces['SOC'].iloc[-1]-.5)<1e-5
            assert traces['meter_balance_residual_kW'].abs().max()<1e-5
            old=pd.read_csv(site_base/'Plots/Cost'/key/'daily_financial_detail.csv');old['Date']=pd.to_datetime(old['Date'])
            old=old[old['Case'].eq('Both')].set_index('Date')
            previous_pd=previous_ncd=0.0
            for date,d in impl.groupby(impl['Interval start'].dt.normalize(),sort=True):
                day=date.strftime('%Y%m%d');mask=pd.to_datetime(traces['Interval start']).dt.normalize().eq(date)
                tr=traces[mask]
                if not is_chained(month):
                    # Retained batches have authoritative cent-rounded daily ledgers;
                    # some optional interval-profit exports were not retained.
                    v={field:float(old.loc[date,field]) for field in components+['Total Revenue']}
                    daily_rows.append({'site':site,'month':month,'forecast':forecast,'Date':date,**v})
                    continue
                wm=pd.read_csv(site_base/'Plots/Solver_RT_Choices'/f'{key}_full'/day/'RT_WM_profit_breakdown_full.csv')
                pdpeak=float(d['PD_Base'].max());ncd=float(d['NCD_Base'].max());pd_rate=3.05 if month>=6 else .63
                v={'WM Revenue':float(wm['profit_sum_all_products'].sum()),'EV Revenue':.4*float(d['Base [kWh]'].sum()),
                   'TOU Cost':-float(np.sum(d['Base_meter [kWh]'].to_numpy()*tr['TOU_$/kWh'].to_numpy())),
                   'PD Cost':-pd_rate*max(pdpeak-previous_pd,0),'NCD Cost':-15.38*max(ncd-previous_ncd,0)}
                v['Total Revenue']=sum(v.values());previous_pd=max(previous_pd,pdpeak);previous_ncd=max(previous_ncd,ncd)
                for field,value in v.items():assert abs(float(old.loc[date,field])-value)<.0052,(site,month,forecast,field,value,old.loc[date,field])
                daily_rows.append({'site':site,'month':month,'forecast':forecast,'Date':date,**v})
            f=pd.DataFrame([r for r in daily_rows if r['site']==site and r['month']==month and r['forecast']==forecast])
            rows.append({'site':site,'month':month,'Scenario':forecast,'days':len(f),**f[components+['Total Revenue']].sum().to_dict()})
            profiles[site,month,forecast]=traces['p_GI_kW'].to_numpy().reshape(-1,96).mean(axis=0)
            passive=traces[['p_load_kW','p_PV_kW','TOU_$/kWh']].to_numpy()
            if reference is not None:assert np.allclose(passive,reference,atol=1e-6,rtol=0)
            reference=passive
            provenance.append({'site':site,'month':month,'forecast':forecast,'dispatch_file':str(path.relative_to(ROOT)),
                'history_batch':'March-onward chronological chain' if is_chained(month) else 'Retained standalone batch'})
        rates=reference[:,2];onpeak=np.isclose(rates,rates.max())
        for label,power in [('Building only',reference[:,0]),('Building + PV',reference[:,0]-reference[:,1])]:
            v={'WM Revenue':0.,'EV Revenue':0.,'TOU Cost':-float(np.sum(power*rates*.25)),
               'PD Cost':-(3.05 if month>=6 else .63)*max(float(power[onpeak].max()),0),
               'NCD Cost':-15.38*max(float(power.max()),0)}
            rows.append({'site':site,'month':month,'Scenario':label,'days':len(expected)//96,**v,'Total Revenue':sum(v.values())})
        profiles[site,month,'Building + PV']=(reference[:,0]-reference[:,1]).reshape(-1,96).mean(axis=0)
staged=pd.DataFrame(rows)
daily=pd.DataFrame(daily_rows)
ev=daily.pivot(index=['site','Date'],columns='forecast',values='EV Revenue')
assert (ev['Perfect']-ev['Persistence']).abs().max()<.02
for filename,table in [('site_resource_addition_financial_comparison_mar_jul_2025.csv',staged),('daily_financial_comparison_mar_jul_2025.csv',daily),('site_monthly_source_provenance_mar_jul_2025.csv',pd.DataFrame(provenance))]:
    table.to_csv(EXPORT/filename,index=False,float_format='%.10f');shutil.copy2(EXPORT/filename,REVIEW/filename)

style={'font.family':'DejaVu Sans','font.size':8,'axes.titleweight':'bold','axes.spines.top':False,'axes.spines.right':False,'legend.frameon':False}
with plt.rc_context(style):
    fig,axes=plt.subplots(5,2,figsize=(9,13.3),layout='constrained')
    order=['Building only','Building + PV','Persistence','Perfect']
    ticklabels=['Building\nonly','Building\n+ PV','24-h MPC\nPersistence','24-h MPC\nPerfect']
    for i,month in enumerate(MONTHS):
        for j,site in enumerate(SITES):
            ax=axes[i,j];d=staged[(staged.site==site)&(staged.month==month)].set_index('Scenario').loc[order]
            pos=np.zeros(4);neg=np.zeros(4)
            for component in components:
                v=d[component].to_numpy()/1000
                ax.bar(np.arange(4),v,bottom=np.where(v>=0,pos,neg),width=.64,color=palette[component],edgecolor='white',lw=.35)
                pos+=np.maximum(v,0);neg+=np.minimum(v,0)
            net=d['Total Revenue'].to_numpy()/1000
            ax.scatter(np.arange(4),net,c='#20242A',s=15,marker='D',zorder=5)
            ax.set_title(f'({chr(97+i*2+j)}) {calendar.month_name[month]} — {site.replace("_"," ")}',loc='left',fontsize=9)
            ax.set_xticks(range(4),ticklabels,fontsize=7);ax.set_ylabel('Financial outcome ($1000)')
            ax.axhline(0,color='#333333',lw=.6);ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True);ax.margins(y=.16)
            ax.yaxis.set_major_locator(MaxNLocator(5))
    handles=[Patch(facecolor=palette[c],label=c) for c in components]+[Line2D([],[],marker='D',ls='',c='#20242A',label='Net financial outcome')]
    fig.suptitle('Monthly Financial Outcomes across Resource Configurations\nNTPLL and Center Hall | March–July 2025',fontsize=13,fontweight='bold')
    fig.legend(handles=handles,loc='outside lower center',ncol=3,fontsize=8)
    save_figure(fig,'fig_17_site_resource_addition_financial_comparison_mar_jul_2025.png')

    fig,axes=plt.subplots(1,2,figsize=(9,4.0),layout='constrained')
    for ax,site in zip(axes,SITES):
        for forecast in FORECASTS:
            d=staged[(staged.site==site)&(staged.Scenario==forecast)].set_index('month').loc[MONTHS]
            ax.plot(MONTHS,d['Total Revenue']/1000,'o-',color=forecast_colors[forecast],lw=1.8,ms=4,label=forecast)
        ax.set_title(site.replace('_',' '),loc='left');ax.set_xticks(MONTHS,[calendar.month_abbr[m] for m in MONTHS]);ax.set_ylabel('Net operating revenue ($1000/month)');ax.grid(axis='y',alpha=.18);ax.margins(y=.15)
    axes[1].legend(loc='best');fig.suptitle('Monthly Net Operating Revenue | Site-specific 24-h MPC\nMarch–July 2025',fontsize=12,fontweight='bold')
    save_figure(fig,'site_monthly_net_revenue_mar_jul_2025.png')

    fig,axes=plt.subplots(5,2,figsize=(9,11.8),sharex=True,sharey='col',layout='constrained')
    for i,month in enumerate(MONTHS):
        for j,site in enumerate(SITES):
            ax=axes[i,j]
            for label in ['Building + PV',*FORECASTS]:
                ax.plot(np.arange(96)/4,profiles[site,month,label],label=label,color=forecast_colors.get(label,'#808992'),ls=':' if label=='Building + PV' else '--' if label=='Persistence' else '-',lw=1.25)
            ax.set_title(f'({chr(97+i*2+j)}) {calendar.month_name[month]} — {site.replace("_"," ")}',loc='left',fontsize=9)
            ax.set_ylabel('Mean grid power (kW)');ax.set_xticks([0,6,12,18,24]);ax.grid(alpha=.18)
    for ax in axes[-1]:ax.set_xlabel('Hour of day')
    h,l=axes[0,0].get_legend_handles_labels();fig.legend(h,l,loc='outside lower center',ncol=3)
    fig.suptitle('Monthly Mean Daily Grid-Power Profiles\nNTPLL and Center Hall | March–July 2025',fontsize=13,fontweight='bold')
    save_figure(fig,'site_monthly_mean_grid_profiles_mar_jul_2025.png')
print('March-July exports complete; source provenance saved; paper text unchanged.')


In [ ]:
# Fig16-style monthly site power specifications, March-July 2025.
from pathlib import Path
import calendar, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
ROOT=Path.cwd().resolve()
STUDY=ROOT/'Results_Rolling/Site_Cases_2025/as_settlement_fix_20260912'
OUT=STUDY/'monthly_runs/cross_month_comparison'
REVIEW=ROOT/'Paper_Figures/financial_comparison/site_cross_month_2025'
LATEX=ROOT/'Latex/paper3.5/figures'
for p in [OUT,REVIEW,LATEX]:p.mkdir(parents=True,exist_ok=True)
META={'NTPLL':(34,55.33+54.88+56.72),'Center_Hall':(38,206.15)}
rows=[]
for site,(ports,pv_proxy) in META.items():
    ev=pd.read_csv(ROOT/f'2025Data/Site_Data_2025/{site}_EV_2025_QC.csv',low_memory=False)
    ev['Interval start']=pd.to_datetime(ev['Interval start'])
    assert ev['Station Name'].nunique()==ports
    for month in range(3,8):
        period=STUDY/'monthly_runs'/f'2025-{month:02d}'
        if month==6 and not (period/'202506_month_Both_FULL_BTM_Both_manifest.json').exists():period=STUDY
        directory=period/'model_outputs'/site/'FULL_BTM/Validation_Traces/2025_PersistenceSessionkWh_PersistenceNumbEV_PerfectatArrival_full'
        traces=pd.concat([pd.read_csv(directory/f'rolling_validation_trace_2025{month:02d}{day:02d}.csv') for day in range(1,calendar.monthrange(2025,month)[1]+1)],ignore_index=True)
        assert len(traces)==calendar.monthrange(2025,month)[1]*96
        e=ev[ev['Interval start'].dt.year.eq(2025)&ev['Interval start'].dt.month.eq(month)]
        peak=pd.to_numeric(e['Interval average demand kW'],errors='raise').groupby(e['Interval start']).sum().max()
        rows.append({'site':site,'month':month,'Gross building peak':float(traces.p_load_kW.max()),
            'PV capacity proxy':pv_proxy,'Observed PV peak':float(traces.p_PV_kW.max()),
            'EVSE model limit':ports*6.656,'Observed EV coincident peak':float(peak),'BESS power':250.,'BESS energy (kWh)':332.,
            'PV proxy definition':'Fixed observed-capacity proxy from reviewed 2024 site study; not nameplate',
            'EV peak definition':'Raw QC interval-average coincident charging demand; not optimized dispatch'})
specs=pd.DataFrame(rows)
name='site_system_power_specifications_mar_jul_2025.csv'
specs.to_csv(OUT/name,index=False,float_format='%.6f');shutil.copy2(OUT/name,REVIEW/name)
metrics=['Gross building peak','PV capacity proxy','Observed PV peak','EVSE model limit','Observed EV coincident peak','BESS power']
labels=['Gross building peak','PV capacity proxy','Observed PV peak','EVSE model limit','Observed EV peak','BESS power']
colors=['#8A8F98','#E9C46A','#2A9D8F','#F4A261','#D95F59','#4C78A8']
with plt.rc_context({'font.family':'DejaVu Sans','font.size':8,'axes.titleweight':'bold','axes.spines.top':False,'axes.spines.right':False}):
    fig,axes=plt.subplots(5,2,figsize=(10,12.7),layout='constrained')
    for i,month in enumerate(range(3,8)):
        for j,site in enumerate(META):
            ax=axes[i,j];row=specs[(specs.site==site)&(specs.month==month)].iloc[0]
            values=[row[m] for m in metrics]
            bars=ax.barh(range(6),values,color=colors,edgecolor='white',lw=.4)
            ax.set_yticks(range(6),labels);ax.invert_yaxis();ax.set_xlabel('Power (kW)')
            ax.set_title(f'({chr(97+2*i+j)}) {calendar.month_name[month]} — {site.replace("_"," ")}',loc='left',fontsize=10)
            limit=float(specs[specs.site==site][metrics].max().max())
            ax.set_xlim(0,limit*1.20);ax.xaxis.set_major_locator(MaxNLocator(5));ax.grid(axis='x',alpha=.20);ax.set_axisbelow(True)
            for bar,value in zip(bars,values):
                ax.text(value+limit*.015,bar.get_y()+bar.get_height()/2,f'{value:,.1f}',va='center',fontsize=8)
    fig.suptitle('Site System Power Specifications and Monthly Observed Peaks\nNTPLL and Center Hall | March–July 2025',fontsize=13,fontweight='bold')
    fig.supxlabel('Fixed BESS: 250 kW / 332 kWh   |   NTPLL: 34 EVSE   |   Center Hall: 38 EVSE\nPV capacity proxy is not nameplate capacity; observed EV peaks precede optimization.',fontsize=8)
    filename='fig_16_site_system_power_specifications_mar_jul_2025.png'
    fig.savefig(OUT/filename,dpi=300,bbox_inches='tight',pad_inches=.08,facecolor='white')
    for p in [REVIEW,LATEX]:shutil.copy2(OUT/filename,p/filename)
    plt.close(fig)
print(OUT/filename)


## SEGAN v3.3 publication-only daily panels
Saved July 1/15 trajectories, common within-site limits, and explicit price/obligation identities. No optimization is run.

In [ ]:
# SEGAN v3.3: publication panels from saved chronological site runs only.
# This cell does not execute optimization or change any dispatch data.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import shutil

ROOT = Path('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024')
RUNS = ROOT / 'Results_Rolling/Site_Cases_2025/as_settlement_fix_20260912/monthly_runs'
OUT = ROOT / 'Paper_Figures/financial_comparison/site_cross_month_2025'
OUT.mkdir(parents=True, exist_ok=True)
DATES = ['20250701', '20250715']
COLORS = dict(Energy='#1f77b4', RU='#2ca02c', RD='#d62728', SP='#9467bd', NSP='#8c564b')
RT_COLORS = dict(Energy='#17becf', RU='#98df8a', RD='#ff9896', SP='#c5b0d5', NSP='#c49c94')
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titlesize':11,
                     'axes.labelsize':10,'legend.fontsize':8,'axes.spines.top':False})

def load_day(site, date, forecast='Perfect'):
    base=RUNS/'2025-07/model_outputs'/site/'FULL_BTM'
    key=f'2025_{forecast}SessionkWh_{forecast}NumbEV_PerfectatArrival_full'
    tr=pd.read_csv(base/'Validation_Traces'/key/f'rolling_validation_trace_{date}.csv',parse_dates=['Interval start']).set_index('Interval start')
    folder=base/'Plots/Solver_RT_Choices'/key/date
    audit=pd.read_csv(folder/'RT_market_interval_audit_with_prices.csv',parse_dates=['Interval start'])
    activity=pd.read_csv(folder/f'Bess_activity_table_{key}.csv')
    impl=pd.read_csv(base/'Dispatch'/f'{key}_implementation.csv',parse_dates=['Interval start']).set_index('Interval start').loc[tr.index]
    assert len(tr)==96 and tr.index.is_unique
    assert np.max(np.abs(tr.p_GI_kW-(tr.p_EV_kW+tr.p_BESS_kW+tr.p_load_kW-tr.p_PV_kW)))<1e-6
    products={s:audit[audit.Product==s].set_index('Interval start').loc[tr.index] for s in COLORS}
    q=sum(v['Signed energy obligation (kW)'].to_numpy() for v in products.values())
    assert np.max(np.abs(q-(tr.p_actual_kW+sum((1 if s!='RD' else -1)*products[s].Alpha.to_numpy()*tr[f'c_{s}_actual_kW'] for s in ['RU','RD','SP','NSP']))))<1e-6
    return tr,products,activity,impl,q

def legend(ax,n=4,other=None):
    h,l=ax.get_legend_handles_labels()
    if other is not None:
        hh,ll=other.get_legend_handles_labels();h+=hh;l+=ll
    ax.legend(h,l,loc='lower left',bbox_to_anchor=(0,1.025,1,0.2),mode='expand',
              ncol=min(n,len(l)),borderaxespad=0,frameon=False,handlelength=1.7,columnspacing=.8)

def bars(ax,x,values):
    pos=np.zeros(96); neg=np.zeros(96)
    for label,v,color in values:
        v=np.asarray(v); bot=np.where(v>=0,pos,neg)
        ax.bar(x,v,width=.23,bottom=bot,color=color,label=label,linewidth=0,alpha=.82)
        pos+=np.maximum(v,0);neg+=np.minimum(v,0)
    return neg,pos

def lighter(color):
    return np.array(mcolors.to_rgb(color))*.52+.48

def panel(site,date):
    tr,prod,act,impl,q=load_day(site,date);x=np.arange(96)/4
    fig,axs=plt.subplots(3,2,figsize=(15.2,10.4),sharex=True)
    fig.subplots_adjust(left=.065,right=.93,bottom=.06,top=.92,hspace=.70,wspace=.25)
    twin=[]
    for j,(ax,title) in enumerate(zip(axs.flat,['(a) Total Expected Price','(b) Raw Prices','(c) Bids','(d) Energy Obligation (α * Capacity)','(e) BESS Action & SOC','(f) Overall System Power'])):
        ax.set_title(title,loc='left',pad=58 if j in (0,1,2,3) else 38,fontweight='bold')
        ax.grid(axis='y',alpha=.17);ax.set_axisbelow(True);ax.set_xlim(0,24);ax.set_xticks(np.arange(0,25,4))
        ax.set_ylabel('Power (kW)' if j>=2 else 'Price ($/kWh)')
    for k in (0,1):
        ax=axs.flat[k]
        for s,v in prod.items():
            for stage,style in [('DA','-'),('RT','--')]:
                if s=='Energy': y=v[f'{stage} LMP ($/kWh)']
                else:
                    y=v[f'{stage} AS capacity price ($/kWh)']
                    if k==0:y=y+(1 if s!='RD' else -1)*v.Alpha*v['RT LMP ($/kWh)']
                label=f'LMP {stage}' if s=='Energy' else f'{"TP" if k==0 else "Raw"} {s} {stage}'
                ax.plot(x,y,color=COLORS[s],ls=style,lw=1.2,label=label)
        a=ax.twinx();a.plot(x,tr['TOU_$/kWh'],color='#00a5a5',ls='-.',lw=1.5,label='TOU');a.set_ylabel('TOU ($/kWh)',color='#008b8b');a.tick_params(axis='y',labelcolor='#008b8b');a.set_ylim(.08,.88);twin.append(a)
        legend(ax,5,other=a)
    bids=[];obls=[]
    for s in ['Energy','RU','SP','NSP','RD']:
        v=prod[s]
        for stage,col in [('DA','DA quantity (kW)'),('RT','RT signed adjustment (kW)')]:
            sign=-1 if s=='RD' else 1
            y=v[col].to_numpy()*sign
            color=COLORS[s] if stage=='DA' else RT_COLORS[s]
            label=f'p_{stage}' if s=='Energy' else f'c_{s}_{stage}'
            ol=f'Obl p_{stage}' if s=='Energy' else f'Obl {s}_{stage}'
            bids.append((label,y,color))
            obls.append((ol,y*(v.Alpha.to_numpy() if s!='Energy' else 1),color))
    bars(axs[1,0],x,bids)
    axs[1,0].plot(x,tr.p_up_bound_kW,color='#e6550d',lw=1.6,label='upper bound')
    axs[1,0].plot(x,-tr.p_down_bound_kW,color='#3182bd',lw=1.6,label='lower bound')
    bars(axs[1,1],x,obls);axs[1,1].plot(x,q,color='black',lw=1.2,label='Net Obligation')
    for ax in axs[1]:ax.axhline(0,color='.4',lw=.7);legend(ax,6)
    ax=axs[2,0]
    # Saved activity rows contain every non-idle interval, rounded to 0.01 kW.
    flow=act.set_index('time').reindex(tr.index.strftime('%H:%M'))
    flow=flow[['p_ch_WM(kW)','p_ch_NWM(kW)','p_dch_WM(kW)','p_dch_NWM(kW)']].fillna(0)
    net=flow.iloc[:,0]+flow.iloc[:,1]-flow.iloc[:,2]-flow.iloc[:,3]
    assert np.max(np.abs(net.to_numpy()-tr.p_BESS_kW.to_numpy()))<=.021
    bars(ax,x,[('BESS ch WM',flow.iloc[:,0],'#1f77b4'),('BESS ch NWM',flow.iloc[:,1],'#17becf'),('BESS dch WM',-flow.iloc[:,2],'#d62728'),('BESS dch NWM',-flow.iloc[:,3],'#ff9896')]);ax.axhline(0,color='.4',lw=.7)
    a=ax.twinx();a.plot(x,tr.SOC*100,color='#2ca02c',lw=1.5,label='SOC');a.set_ylim(0,100);a.set_ylabel('SOC (%)',color='#2ca02c');a.tick_params(axis='y',labelcolor='#2ca02c');twin.append(a);legend(ax,3,other=a)
    ax=axs[2,1]
    ax.step(x,tr.p_EV_kW,where='post',color='#1f77b4',label='p_EV');ax.step(x,tr.baseline_kW,where='post',color='#6c757d',ls='--',label='Baseline')
    ax.fill_between(x,tr.p_EV_kW,tr.baseline_kW,where=tr.p_EV_kW>=tr.baseline_kW,step='post',color='#f4a261',alpha=.2,label='pEV > Baseline')
    ax.fill_between(x,tr.p_EV_kW,tr.baseline_kW,where=tr.p_EV_kW<tr.baseline_kW,step='post',color='#90caf9',alpha=.2,label='pEV < Baseline')
    ax.step(x,tr.p_BESS_kW,where='post',color='#d62728',label='p_BESS_net')
    ax.step(x,tr.p_GI_kW,where='post',color='#2ca02c',label='p_GI')
    ax.plot(x,impl.NCD_Base,color='#ff9f1c',ls='--',label='NCD Threshold');ax.plot(x,impl.PD_Base,color='#e76f51',ls='--',label='PD Threshold')
    ax.set_ylabel('Net Power (kW)');legend(ax,4)
    for ax in axs[2]:ax.set_xlabel('Hour of day')
    return fig,list(axs.flat),twin

def common_limits(axes):
    lo=min(a.dataLim.ymin for a in axes);hi=max(a.dataLim.ymax for a in axes)
    margin=max((hi-lo)*.06,.001)
    for a in axes:a.set_ylim(lo-margin,hi+margin)

for site in ['NTPLL','Center_Hall']:
    figs=[panel(site,d) for d in DATES]
    for i in range(6):common_limits([f[1][i] for f in figs])
    # SOC keeps the same 0--100% scale on both dates.
    for date,(fig,_,_) in zip(DATES,figs):
        fig.savefig(OUT/f'{site.lower()}_perfect_daily_6panel_{date}_paper.png',dpi=300,bbox_inches='tight',facecolor='white');plt.close(fig)
print('Publication-only July figures generated from saved dispatch; no optimizer invoked.')

# Synchronize only the current manuscript copies; retain all source figures.
paper_dir = ROOT / 'Latex/paper3.3'
if (paper_dir / 'main_segan_v3.3.tex').exists():
    import re
    for name in re.findall(r'\\includegraphics(?:\[[^]]*\])?\{([^}]+)\}', (paper_dir / 'main_segan_v3.3.tex').read_text()):
        suffix = re.sub(r'^fig_\d+_', '', name)
        candidate = OUT / suffix
        if candidate.exists():
            shutil.copy2(candidate, paper_dir / 'figures' / name)


In [ ]:
# Paired weekly sensitivity figures: rows are forecasts, columns are financial views.
# Reads validated saved results only. No optimization or manuscript text edit.
def plot_weekly_sensitivity(root=None):
    from pathlib import Path
    import shutil
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    project=Path('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024')
    root=Path(root) if root else project/'Sensitivity_Results/Rolling_24h/site_week_20250301_07_as_settlement_fix_20260912'
    tables=root/'tables';data=pd.read_csv(tables/'weekly_financial_comparison.csv').set_index('case_id',drop=False)
    parts=['WM Revenue','EV Revenue','TOU Cost','PD Cost','NCD Cost']
    if data.index.duplicated().any() or not data.PASS.all():raise ValueError('Duplicate or unvalidated cases')
    if (data['Total Revenue']-data[parts].sum(axis=1)).abs().max()>.031*7:raise ValueError('Financial reconciliation failed')
    forecasts=['Persistence','Perfect']
    orders={
      'factor':['reference','factor_000','factor_050','factor_100','factor_random_11','factor_random_22','factor_random_33','factor_random50_44'],
      'bess':['reference','bess_power2','bess_energy2','bess_both2'],
      'eta':['reference','eta_080','eta_050'],
      'market':['market_retail','market_energy','reference']}
    def case_id(scenario,forecast):
        return ('reference' if forecast=='Persistence' else 'perfect_24h') if scenario=='reference' else scenario+('_perfect' if forecast=='Perfect' else '')
    expected=[case_id(s,f) for order in orders.values() for s in order for f in forecasts]
    missing=sorted(set(expected)-set(data.index))
    if missing:raise ValueError('Paired figures require complete results: '+str(missing))
    checks=[]
    for scenario in sorted(set(sum(orders.values(),[]))):
        a=data.loc[case_id(scenario,'Persistence')];b=data.loc[case_id(scenario,'Perfect')]
        for field in ['eta','P_BESS_kW','E_BESS_kWh','alpha_SP','alpha_NSP','market_mode','baseline_policy','random_seed']:
            av,bv=a[field],b[field]
            equal=(pd.isna(av) and pd.isna(bv)) or av==bv
            checks.append({'scenario':scenario,'parameter':field,'PASS':bool(equal)})
    check=pd.DataFrame(checks);check.to_csv(tables/'paired_parameter_validation.csv',index=False)
    if not check.PASS.all():raise ValueError('Forecast pair has different scenario parameters')
    colors={'WM Revenue':'#247BA0','EV Revenue':'#E99532','TOU Cost':'#43A87B','PD Cost':'#9564AD','NCD Cost':'#D95861'}
    forecast_colors={'Persistence':'#247BA0','Perfect':'#E99532'}
    def label(r,family):
        if family=='factor':
            if pd.notna(r.random_seed):return f'Random\nmean {r.alpha_SP:g}\nseed {int(r.random_seed)}'
            return f'Constant\n{r.alpha_SP:g}'
        if family=='bess':return f'{r.P_BESS_kW:g} kW\n{r.E_BESS_kWh:g} kWh'
        if family=='eta':return f'Minimum service\n{100*r.eta:g}%'
        return {'full':'Retail + energy\n+ AS','energy_only':'Retail + energy','retail_only':'Retail only'}[r.market_mode]
    titles={'factor':'SP/NSP attenuation factors','bess':'BESS power and energy capacity','eta':'Minimum EV charging service','market':'Value of joint market participation'}
    out=root/'figures';destinations=[project/'Paper_Figures/weekly_sensitivity',project/'Latex/paper3.5/figures']
    for d in [out,*destinations]:d.mkdir(parents=True,exist_ok=True)
    saved=[];all_rows=[];wide_rows=[]
    def save(fig,name):
        path=out/name;fig.savefig(path,dpi=300,bbox_inches='tight',facecolor='white');plt.close(fig)
        for d in destinations:shutil.copy2(path,d/name)
        saved.append(path)
    with plt.rc_context({'font.family':'DejaVu Sans','font.size':10,'axes.titleweight':'bold','axes.spines.top':False,'axes.spines.right':False}):
        for number,family in [(21,'factor'),(22,'bess'),(23,'eta'),(24,'market')]:
            order=orders[family];x=np.arange(len(order));fig,axes=plt.subplots(2,2,figsize=(15.4 if family=='factor' else 13.6,9.5),sharey='col')
            gains=[];foot=[];family_rows=[]
            for row,forecast in enumerate(forecasts):
                subset=data.loc[[case_id(s,forecast) for s in order]].copy()
                base=data.loc[case_id('market_retail' if family=='market' else 'reference',forecast)]
                full=data.loc[case_id('reference',forecast)]
                gain=subset['Total Revenue'].to_numpy()-float(base['Total Revenue']);gains.extend(gain)
                names=[label(r,family) for _,r in subset.iterrows()]
                left,right=axes[row];left.bar(x,gain,width=.62,color=forecast_colors[forecast],zorder=3)
                for k,v in enumerate(gain):
                    text=f'{v:+,.2f}'
                    if family=='eta':text+=f"\nDelivered {100*subset.iloc[k].EV_delivered_kWh/full.EV_delivered_kWh:.1f}%"
                    left.annotate(text,(k,v),xytext=(0,6 if v>=0 else -6),textcoords='offset points',ha='center',va='bottom' if v>=0 else 'top',fontsize=9)
                positive=np.zeros(len(order));negative=positive.copy()
                for component in parts:
                    delta=subset[component].to_numpy()-float(base[component])
                    right.bar(x,delta,bottom=np.where(delta>=0,positive,negative),width=.62,color=colors[component],label=component,zorder=3)
                    positive+=np.maximum(delta,0);negative+=np.minimum(delta,0)
                right.scatter(x,gain,c='#222222',marker='D',s=28,label='Net revenue',zorder=5)
                reftext='retail only' if family=='market' else 'full-market reference'
                left.set_title(f'({"a" if row==0 else "c"}) {forecast}: net revenue change',loc='left',pad=13)
                right.set_title(f'({"b" if row==0 else "d"}) {forecast}: decomposition of change',loc='left',pad=13)
                left.set_ylabel('Net revenue change ($)');right.set_ylabel('Revenue / signed-cost change ($)')
                for ax in (left,right):
                    ax.set_xticks(x,names);ax.tick_params(axis='x',labelsize=9,pad=6);ax.axhline(0,c='#444',lw=.8);ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True)
                foot.append(f"{forecast} {reftext}: USD {base['Total Revenue']:,.2f}")
                for scenario,(_,r),delta in zip(order,subset.iterrows(),gain):
                    rr={'family':family,'scenario':scenario,'forecast':forecast,'case_id':r.case_id,'scenario_label':label(r,family).replace('\n',' / '),'comparison_case_id':base.case_id,'Net revenue change ($)':float(delta),'EV delivered (kWh)':float(r.EV_delivered_kWh),'Actual service (%)':100*float(r.EV_delivered_kWh)/float(full.EV_delivered_kWh),'eta':float(r.eta)}
                    rr.update({c:float(r[c]) for c in ['Total Revenue',*parts,'WM Energy Revenue','WM Capacity Revenue']});family_rows.append(rr);all_rows.append(rr)
            lo=min(0,min(gains));hi=max(0,max(gains));span=max(hi-lo,1)
            axes[0,0].set_ylim(lo-.22*span,hi+.30*span)
            lo=min(a.dataLim.ymin for a in axes[:,1]);hi=max(a.dataLim.ymax for a in axes[:,1]);span=max(hi-lo,1)
            axes[0,1].set_ylim(lo-.12*span,hi+.18*span)
            handles,names=axes[0,1].get_legend_handles_labels();fig.legend(handles,names,loc='lower center',bbox_to_anchor=(.5,.055),ncol=6,frameon=False,fontsize=10)
            fig.suptitle(f'NTPLL | {titles[family]} | 01–07 March 2025',fontsize=16,fontweight='bold',y=.98)
            fig.text(.065,.025,' | '.join(foot)+'. Each forecast uses its own comparison baseline.',fontsize=9)
            fig.subplots_adjust(left=.065,right=.985,top=.90,bottom=.18,hspace=.68,wspace=.23)
            save(fig,f'fig_{number:02d}_weekly_{family}_sensitivity.png')
            fd=pd.DataFrame(family_rows);fd.to_csv(tables/f'paired_{family}_financial_detail.csv',index=False)
            for scenario in order:
                dd=fd[fd.scenario.eq(scenario)].set_index('forecast');wr={'family':family,'scenario':scenario,'setting':dd.iloc[0].scenario_label}
                for forecast in forecasts:
                    for col in ['Total Revenue','Net revenue change ($)','WM Revenue','Actual service (%)']:
                        wr[f'{forecast}: {col}']=dd.loc[forecast,col]
                wide_rows.append(wr)
        pd.DataFrame(all_rows).to_csv(tables/'paired_sensitivity_financial_detail.csv',index=False)
        pd.DataFrame(wide_rows).to_csv(tables/'paired_sensitivity_paper_table.csv',index=False)
        # Keep the standard oracle only, with its benchmark scope explicit.
        oracle=pd.read_csv(tables/'oracle_financial_comparison.csv');oracle=oracle[oracle.case_id.eq('oracle')]
        bench=pd.concat([data.loc[['benchmark_persistence','benchmark_perfect']].reset_index(drop=True),oracle],ignore_index=True)
        net0=float(bench.iloc[0]['Total Revenue']);gain=bench['Total Revenue'].to_numpy()-net0;x=np.arange(len(bench))
        fig,ax=plt.subplots(figsize=(8.5,4.8));ax.bar(x,gain,color=['#247BA0','#E99532','#43A87B'],width=.6)
        for k,v in enumerate(gain):ax.annotate(f'{v:+,.2f}',(k,v),xytext=(0,6),textcoords='offset points',ha='center')
        if len(oracle):ax.scatter([2],[float(oracle.iloc[0].objective_upper_bound)-net0],c='#222',marker='_',s=180,label='Oracle solver upper bound (1% tolerance)');ax.legend(frameon=False,fontsize=9)
        ax.set_xticks(x,['24-h Persistence','24-h Perfect','Full-horizon oracle']);ax.set_ylabel('Net revenue gain vs matched Persistence ($)');ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True);ax.margins(y=.18)
        fig.suptitle('NTPLL | Matched-baseline horizon benchmark | 01–07 March 2025',fontsize=13,fontweight='bold')
        fig.text(.06,.015,'Standard oracle is a benchmark, not an unconditional upper bound for forecast-driven DA offers.',fontsize=8)
        fig.tight_layout(rect=(0,.05,1,.94));save(fig,'fig_26_weekly_oracle_forecast_comparison.png')
    print('Paired figures and paper-ready tables saved:',*saved,sep='\n')
    return saved

import os as _weekly_plot_os
if _weekly_plot_os.environ.get('RUN_WEEKLY_SENSITIVITY_PLOTS','0')=='1':
    plot_weekly_sensitivity()


In [ ]:
# Publication-style weekly six-panel figures reuse the approved renderer above.
# Only existing completed case PNGs are refreshed; no optimization is performed.
def render_weekly_publication_panels(root=None, case_ids=None):
    from pathlib import Path
    import json
    import shutil
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    project=Path('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024')
    study=Path(root) if root is not None else project/'Sensitivity_Results/Rolling_24h/site_week_20250301_07_as_settlement_fix_20260912'
    notebook=json.loads((project/'paper_financial_comparison_plots.ipynb').read_text())
    candidates=[''.join(c.get('source',[])) for c in notebook['cells'] if ''.join(c.get('source',[])).startswith('# SEGAN v3.3: publication panels from saved chronological site runs only.')]
    if len(candidates)!=1:raise ValueError('Expected exactly one approved publication-renderer source cell')
    source=candidates[0]
    marker="\nfor site in ['NTPLL','Center_Hall']:"
    if source.count(marker)!=1:raise ValueError('Publication renderer boundary changed; review before running')
    # Execute definitions only: the July rendering loop and synchronization are excluded.
    definitions=source.split(marker,1)[0]
    ns={'__name__':'weekly_publication_renderer'}
    saved=[];destinations=[study/'figures',project/'Paper_Figures/weekly_sensitivity',project/'Latex/paper3.5/figures']
    with plt.rc_context():
        exec(compile(definitions,'approved_publication_definitions','exec'),ns)
        for complete in sorted((study/'cases').glob('*/COMPLETE.json')):
            manifest=json.loads(complete.read_text())
            if case_ids is not None and manifest['case_id'] not in case_ids:continue
            if not manifest.get('validation',{}).get('PASS',False):continue
            case_dir=complete.parent
            key=manifest['forecast_key']+'_'+manifest['mode_in_files']
            def load_weekly_day(site,date,forecast='Perfect'):
                tr=pd.read_csv(case_dir/'Validation_Traces'/key/f'rolling_validation_trace_{date}.csv',parse_dates=['Interval start']).set_index('Interval start')
                folder=case_dir/'Plots/Solver_RT_Choices'/key/date
                audit=pd.read_csv(folder/'RT_market_interval_audit_with_prices.csv',parse_dates=['Interval start'])
                activity=pd.read_csv(folder/f'Bess_activity_table_{key}.csv')
                impl=pd.read_csv(case_dir/'Dispatch'/f'{key}_implementation.csv',parse_dates=['Interval start']).set_index('Interval start').loc[tr.index]
                if len(tr)!=96 or not tr.index.is_unique:raise ValueError(f'Incomplete daily trace {case_dir} {date}')
                if np.max(np.abs(tr.p_GI_kW-(tr.p_EV_kW+tr.p_BESS_kW+tr.p_load_kW-tr.p_PV_kW)))>=1e-5:raise ValueError('Meter identity failed')
                products={p:audit[audit.Product.eq(p)].set_index('Interval start').loc[tr.index] for p in ns['COLORS']}
                q=sum(v['Signed energy obligation (kW)'].to_numpy() for v in products.values())
                expected=tr.p_actual_kW+sum((1 if p!='RD' else -1)*products[p].Alpha.to_numpy()*tr[f'c_{p}_actual_kW'] for p in ['RU','RD','SP','NSP'])
                if np.max(np.abs(q-expected))>=1e-5:raise ValueError('Signed energy obligation failed')
                return tr,products,activity,impl,q
            ns['load_day']=load_weekly_day
            dates=[d.replace('-','') for d in manifest['dates'] if d in ['2025-03-01','2025-03-07']]
            figures=[]
            try:
                for date in dates:figures.append((date,ns['panel'](manifest.get('site','NTPLL'),date)))
                for index in range(6):ns['common_limits']([axes[index] for _,(_,axes,_) in figures])
                for date,(fig,axes,twins) in figures:
                    folder=case_dir/'Plots/Solver_RT_Choices'/key/date
                    path=folder/f'RT_6panel_{key}.png'
                    if not path.exists():raise FileNotFoundError(f'Expected existing case figure to refresh: {path}')
                    fig.savefig(path,dpi=300,bbox_inches='tight',facecolor='white');saved.append(path)
                    plt.close(fig)
            finally:
                for _,(fig,_,_) in figures:plt.close(fig)
    print(f'Refreshed {len(saved)} existing weekly six-panel PNGs at 300 dpi; per-case outputs only.')
    return saved

import os as _weekly_panels_os
if _weekly_panels_os.environ.get('RUN_WEEKLY_PUBLICATION_PANELS','0')=='1':
    render_weekly_publication_panels()
